# Как устроен парсер name

Парсер извлекает из названия товара структурированные атрибуты. В одном name одновременно могут находиться бренд, тип товара, модель, цвет, материал, размер, артикул и различные физические характеристики.

Для всех этих атрибутов используется не один общий метод, а две независимые части:

* NER-модель — для атрибутов, значение которых зависит от контекста.
* Алгоритмический парсер — для физических величин и размеров.

Такое разделение оказалось практичнее: бренд или модель трудно надёжно определить только по формальным правилам, а конструкции вроде «39 см» или «50x40x30 см» хорошо описываются обычными правилами и регулярными выражениями.

## NER

NER извлекает следующие классы:

* бренд
* тип
* модель
* материал
* цвет
* размер
* назначение
* артикул

Например, в названии

redmond мультиварка rmc-pm504

модель должна определить:

redmond — бренд  
мультиварка — тип  
rmc-pm504 — модель  

В основе используется rubert-tiny2, у которого оставлены первые два Transformer-слоя. Полная модель для коротких товарных названий избыточна, поэтому сокращённая версия заметно быстрее и при этом сохраняет нужный контекст.

Текст сначала токенизируется. Одно исходное слово при этом может разбиться на несколько subword-токенов. Их представления объединяются attention pooling, после чего классификатор определяет класс уже для целого слова.

После классификации соседние слова одного класса объединяются в одну сущность. Например, несколько слов подряд, распознанных как модель, возвращаются как одна модель, а не как отдельные слова.

На последнем этапе результат переводится обратно в позиции исходного name, поэтому для каждой сущности известны её значение, класс, начало и конец в исходной строке.

## Парсер физических величин

Физические величины вынесены в отдельный алгоритмический парсер.

Основная конструкция здесь достаточно формальная: число и единица измерения. Парсер находит такие фрагменты, приводит разные варианты написания единиц к единому виду и определяет, какой физический атрибут найден.

Например:

диаметр 39 сантиметров

превращается в:

диаметр, см: 39  

Для единиц измерения используется набор алиасов. Поэтому «см», «сантиметр» и «сантиметров» приводятся к одной единице «см».

Если перед величиной явно написано название характеристики, используется оно. Поэтому «диаметр 39 см» распознаётся именно как диаметр, а не просто как некоторый размер.

Отдельно обрабатываются многомерные размеры.

50x40 см интерпретируется как:

длина, см: 50  
ширина, см: 40  

50x40x30 см интерпретируется как:

длина, см: 50  
ширина, см: 40  
высота, см: 30  

Поддерживаются разные варианты разделителя между числами: латинская x, русская х, знак × и звёздочка.

После распознавания многомерного размера этот участок текста исключается из обычного поиска физических величин. Это нужно, чтобы из записи 50x40 см дополнительно не извлекалось отдельное значение 40 см.

# Важные замечания

Класс «модель» достаточно широкий. Это связано с исходной разметкой карточек: в него могут попадать не только классические названия моделей, но и серии, версии и другие похожие значения.

В одном name может находиться несколько сущностей одного класса. Например, несколько моделей, цветов или артикулов. Поэтому NER не ограничивается одним значением на класс.

Алгоритмический парсер проще NER, но сильнее зависит от заранее заданных правил. Если появляется новая единица измерения или новый способ записи величины, его нужно добавить в список поддерживаемых вариантов.



In [2]:
# Если pymorphy3 ещё не установлен:

!pip install pint
!pip install -q pymorphy3 ruwordnet
!ruwordnet download

from functools import lru_cache
import numpy as np
import pymorphy3
import torch
import pandas as pd
import json
import re
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModel,
    AutoTokenizer,
)

from transformers.modeling_outputs import (
    TokenClassifierOutput,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 4.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 95.3 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.
google-adk 1.29.0 requires sqlalchemy<3.0.0,>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.
downloading a ruwordnet model from https://github.com/avidale/python-ruwordnet/releases/download/0.0.4/ruwordnet-2021.db


## WordNERModel

Нейросетевая модель для word-level NER по названию товара.

Что делает:

- кодирует название через сокращённый BERT;
- объединяет subword-токены в представления слов через attention pooling;
- учитывает контекст между словами через Transformer Encoder;
- классифицирует каждое слово по типу атрибута.

Выделяемые классы: бренд, тип, модель, материал, цвет, размер, назначение, артикул.

In [3]:
class WordNERModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        num_attention_heads=4,
        pos_dim=16,
        max_subwords_per_word=12,
        attention_hidden=32,
        sequence_heads=4,
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        # Оставляем первые 2 Transformer-слоя.
        self.bert.encoder.layer = nn.ModuleList(
            self.bert.encoder.layer[:2]
        )
        self.bert.config.num_hidden_layers = 2

        hidden_size = self.bert.config.hidden_size

        self.hidden_size = hidden_size
        self.num_attention_heads = num_attention_heads
        self.max_subwords_per_word = max_subwords_per_word

        self.subword_pos_embedding = nn.Embedding(
            max_subwords_per_word,
            pos_dim,
        )

        self.word_attention = nn.Sequential(
            nn.Linear(
                hidden_size + pos_dim,
                attention_hidden,
            ),
            nn.Tanh(),
            nn.Linear(
                attention_hidden,
                num_attention_heads,
            ),
        )

        self.word_projection = nn.Sequential(
            nn.Linear(
                hidden_size * num_attention_heads,
                hidden_size,
            ),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        # Новый word-level contextual layer.
        self.sequence_encoder = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=sequence_heads,
            dim_feedforward=hidden_size * 2,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes,
        )

    def _pool_words(
        self,
        hidden,
        word_ids,
        subword_positions,
        max_words=None,
    ):
        """Attention-pooling subword'ов в слова без построения большого [batch, words, heads, tokens] тензора."""
    
        batch_size, n_tokens, hidden_size = (
            hidden.shape
        )
    
        if max_words is None:
            max_words = (
                int(word_ids.max().item())
                + 1
            )
    
        positions = subword_positions.clamp(
            max=self.max_subwords_per_word - 1
        )
    
        pos_emb = self.subword_pos_embedding(
            positions
        )
    
        attention_features = torch.cat(
            [hidden, pos_emb],
            dim=-1,
        )
    
        # [B, T, H]
        attention_scores = self.word_attention(
            attention_features
        )
    
        n_heads = attention_scores.size(-1)
    
        # CLS/SEP/padding имеют word_id = -1.
        valid = word_ids >= 0
        valid_flat = valid.reshape(-1)
    
        # Только реально принадлежащие словам subwords.
        valid_word_ids = (
            word_ids
            .reshape(-1)[valid_flat]
        )
    
        valid_hidden = (
            hidden
            .reshape(-1, hidden_size)[valid_flat]
        )
    
        valid_scores = (
            attention_scores
            .reshape(-1, n_heads)[valid_flat]
        )
    
        # Уникальный group id для каждой пары:
        # (batch_index, word_index).
        batch_ids = (
            torch.arange(
                batch_size,
                device=hidden.device,
            )[:, None]
            .expand(-1, n_tokens)
            .reshape(-1)[valid_flat]
        )
    
        group_ids = (
            batch_ids * max_words
            + valid_word_ids
        )
    
        n_groups = (
            batch_size * max_words
        )
    
        pooled_heads = []
    
        # Heads всего 4, поэтому маленький цикл дешевле
        # огромного [B, W, H, T] промежуточного тензора.
        for head in range(n_heads):
            scores = valid_scores[:, head]
    
            # Stable softmax: max внутри каждого слова.
            group_max = torch.full(
                (n_groups,),
                -torch.inf,
                dtype=scores.dtype,
                device=hidden.device,
            )
    
            group_max.scatter_reduce_(
                0,
                group_ids,
                scores,
                reduce="amax",
                include_self=True,
            )
    
            weights = torch.exp(
                scores
                - group_max[group_ids]
            )
    
            group_sum = torch.zeros(
                n_groups,
                dtype=weights.dtype,
                device=hidden.device,
            )
    
            group_sum.scatter_add_(
                0,
                group_ids,
                weights,
            )
    
            weights = (
                weights
                / group_sum[group_ids]
                .clamp_min(1e-8)
            )
    
            # Взвешенная сумма subword embeddings
            # отдельно для каждого слова.
            pooled = torch.zeros(
                (
                    n_groups,
                    hidden_size,
                ),
                dtype=hidden.dtype,
                device=hidden.device,
            )
    
            pooled.index_add_(
                0,
                group_ids,
                valid_hidden
                * weights[:, None],
            )
    
            pooled_heads.append(
                pooled.view(
                    batch_size,
                    max_words,
                    hidden_size,
                )
            )
    
        # [B,W,H,D] → [B,W,H*D]
        pooled = torch.stack(
            pooled_heads,
            dim=2,
        ).flatten(
            start_dim=2
        )
    
        return self.word_projection(
            pooled
        )
        
    def forward(
        self,
        input_ids,
        attention_mask,
        word_ids,
        subword_positions,
        token_type_ids=None,
        labels=None,
        **kwargs,
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        hidden = outputs.last_hidden_state

        word_hidden = self._pool_words(
            hidden,
            word_ids,
            subword_positions,
            max_words=(
                labels.size(1)
                if labels is not None
                else None
            ),
        )

        # True = padding, его attention игнорирует.
        word_padding_mask = (
            torch.arange(
                word_hidden.size(1),
                device=word_hidden.device,
            )[None, :]
            >
            word_ids.max(dim=1).values[:, None]
        )

        word_hidden = self.sequence_encoder(
            word_hidden,
            src_key_padding_mask=word_padding_mask,
        )

        logits = self.classifier(
            word_hidden
        )

        loss = None

        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1),
                ),
                labels.reshape(-1),
                ignore_index=-100,
            )

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
        )

    @classmethod
    def from_pretrained_dir(
        cls,
        model_dir,
        num_classes,
        device="cuda",
    ):
        """Создаёт WordNERModel из config + model.pt в указанной директории и сразу переводит её в eval mode."""
    
        with open(
            f"{model_dir}/word_ner_config.json",
            encoding="utf-8",
        ) as f:
            config = json.load(f)
    
        model = cls(
            model_name=config["model_name"],
            num_classes=num_classes,
            num_attention_heads=config["num_attention_heads"],
            pos_dim=config["pos_dim"],
            max_subwords_per_word=config["max_subwords_per_word"],
            attention_hidden=config["attention_hidden"],
            sequence_heads=config.get(
                "sequence_heads",
                4,
            ),
        )
    
        state_dict = torch.load(
            f"{model_dir}/model.pt",
            map_location="cpu",
            weights_only=True,
        )
    
        missing, unexpected = model.load_state_dict(
            state_dict,
            strict=False,
        )
    
        if missing or unexpected:
            print("Missing:", missing)
            print("Unexpected:", unexpected)
    
        return (
            model
            .to(device)
            .eval()
        )

## NerPostprocessor

Полный пайплайн NER-инференса и постобработки предсказаний модели.

Что делает:

- токенизирует названия и сопоставляет subword-токены исходным словам;
- запускает `WordNERModel`;
- объединяет соседние слова одного класса в сущности;
- удаляет явно некорректные сущности;
- проверяет и при необходимости исправляет классы через семантические кластеры;
- расширяет сущности по морфологическим правилам;
- возвращает итоговый словарь извлечённых атрибутов.

Поддерживает обработку одной строки и батча.

In [4]:


class NerPostprocessor:
    """
    Полный быстрый inference + postprocessing:
    NER → runs → rule cleanup → semantic clusters → span expansion → итоговые attributes.
    """

    WORD_RE = re.compile(
        r"\w+(?:[./+\-']\w+)*",
        flags=re.UNICODE,
    )

    PREPOSITIONS = {
        "для", "от", "на", "с", "со", "из", "по",
        "к", "в", "во", "под", "над", "при", "без",
    }

    IMPOSSIBLE_STANDALONE = PREPOSITIONS | {
        "и", "или", "а", "но",
    }


    def __init__(
        self,
        model,
        tokenizer,
        cluster_centers_path,
        class_names,
        device,
        max_length=100,
        use_amp=True,
    ):
        """Сохраняет модель и константы, загружает semantic centers и заранее создаёт все тензоры порогов для быстрого inference."""

        self.model = model
        self.tokenizer = tokenizer
        self.class_names = class_names
    
        self.label2id = {
            label: i
            for i, label in enumerate(class_names)
        }
    
        self.id2label = {
            i: label
            for label, i in self.label2id.items()
        }
  
        self.device = torch.device(device)
        self.max_length = max_length

        self.use_amp = (
            use_amp
            and self.device.type == "cuda"
        )

        self.O_ID = self.label2id["O"]
        self.TYPE_ID = self.label2id["тип"]
        self.ARTICLE_ID = self.label2id["артикул"]

        self.NO_PURE_NUMBER = {
            self.label2id["бренд"],
            self.label2id["тип"],
            self.label2id["материал"],
            self.label2id["цвет"],
            self.label2id["назначение"],
        }

        self.RELABEL_TARGETS = {
            self.label2id["цвет"],
            self.label2id["материал"],
            self.label2id["бренд"],
            self.label2id["модель"],
            self.label2id["артикул"],
        }

        relabel_min_sim = {
            self.label2id["цвет"]: 0.72,
            self.label2id["материал"]: 0.62,
            self.label2id["бренд"]: 0.74,
            self.label2id["модель"]: 0.78,
            self.label2id["артикул"]: 0.82,
        }

        relabel_min_gap = {
            self.label2id["цвет"]: 0.20,
            self.label2id["материал"]: 0.16,
            self.label2id["бренд"]: 0.20,
            self.label2id["модель"]: 0.25,
            self.label2id["артикул"]: 0.30,
        }

        self.RELABEL_MIN_DOMINANCE = 0.06
        self.TYPE_REJECT_GAP = 0.24
        self.GENERAL_REJECT_GAP = 0.20
        self.LOW_OWN_SIM = 0.40
        self.LOW_OWN_REJECT_GAP = 0.12

        n_classes = len(class_names)

        # Создаём один раз, а не на каждом батче.
        self.min_sim = torch.full(
            (n_classes,),
            torch.inf,
            device=self.device,
        )

        self.min_gap = torch.full(
            (n_classes,),
            torch.inf,
            device=self.device,
        )

        for class_id, value in relabel_min_sim.items():
            self.min_sim[class_id] = value

        for class_id, value in relabel_min_gap.items():
            self.min_gap[class_id] = value

        self.relabel_target_mask = torch.zeros(
            n_classes,
            dtype=torch.bool,
            device=self.device,
        )

        self.relabel_target_mask[
            list(self.RELABEL_TARGETS)
        ] = True

        centers = torch.load(
            cluster_centers_path,
            map_location="cpu",
            weights_only=True,
        )
        
        center_parts = []
        center_class_ids = []
        
        for label, values in centers.items():
            if (
                label not in self.label2id
                or len(values) == 0
            ):
                continue
        
            class_id = self.label2id[label]
        
            normalized = F.normalize(
                values.float(),
                p=2,
                dim=1,
            )
        
            center_parts.append(normalized)
        
            center_class_ids.extend(
                [class_id] * len(normalized)
            )
        
        # Все центры физически лежат одним тензором,
        # но принадлежность каждого центра классу сохраняется.
        self.all_centers = torch.cat(
            center_parts,
            dim=0,
        ).to(self.device)
        
        self.center_class_ids = torch.tensor(
            center_class_ids,
            dtype=torch.long,
            device=self.device,
        )
        self.morph = pymorphy3.MorphAnalyzer()

        self.model.eval()


    @lru_cache(maxsize=100_000)
    def _pos(self, word):
        """Возвращает POS наиболее вероятного морфологического разбора слова; результаты кэшируются между всеми последующими карточками."""

        return self.morph.parse(
            word.casefold()
        )[0].tag.POS


    def _get_word_spans(self, text):
        """Возвращает границы исходных слов в строке, используя ту же сегментацию слов, что применялась при обучении модели."""

        return [
            (m.start(), m.end())
            for m in self.WORD_RE.finditer(text)
        ]


    def _align_tokens_to_words(
        self,
        offsets_batch,
        word_spans_batch,
    ):
        """Батчево сопоставляет subword-токены исходным словам и вычисляет позицию каждого subword внутри слова без Python-цикла по токенам."""

        batch_size, max_tokens, _ = (
            offsets_batch.shape
        )

        word_ids = np.full(
            (batch_size, max_tokens),
            -1,
            dtype=np.int32,
        )

        subword_positions = np.zeros(
            (batch_size, max_tokens),
            dtype=np.int32,
        )

        for b, (offsets, spans) in enumerate(
            zip(
                offsets_batch,
                word_spans_batch,
            )
        ):
            if not spans:
                continue

            spans = np.asarray(
                spans,
                dtype=np.int32,
            )

            token_starts = offsets[:, 0]
            token_ends = offsets[:, 1]

            word_starts = spans[:, 0]
            word_ends = spans[:, 1]

            candidates = np.searchsorted(
                word_ends,
                token_starts,
                side="right",
            )

            safe = np.minimum(
                candidates,
                len(spans) - 1,
            )

            valid = (
                (token_starts != token_ends)
                & (candidates < len(spans))
                & (
                    token_starts
                    < word_ends[safe]
                )
                & (
                    token_ends
                    > word_starts[safe]
                )
            )

            token_idx = np.flatnonzero(valid)

            if not token_idx.size:
                continue

            ids = candidates[valid]

            word_ids[
                b,
                token_idx,
            ] = ids

            new_word = np.r_[
                True,
                ids[1:] != ids[:-1],
            ]

            positions = np.arange(
                ids.size,
                dtype=np.int32,
            )

            group_starts = np.maximum.accumulate(
                np.where(
                    new_word,
                    positions,
                    0,
                )
            )

            subword_positions[
                b,
                token_idx,
            ] = (
                positions
                - group_starts
            )

        return (
            word_ids,
            subword_positions,
        )


    @torch.inference_mode()
    def _forward(
        self,
        input_ids,
        attention_mask,
        word_ids,
        subword_positions,
        token_type_ids=None,
    ):
        """Выполняет реальный forward текущей WordNERModel и одновременно сохраняет pooled word embeddings для semantic cluster проверки без второго BERT."""

        with torch.autocast(
            device_type=self.device.type,
            dtype=torch.float16,
            enabled=self.use_amp,
        ):
            outputs = self.model.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

            hidden = outputs.last_hidden_state

            word_lengths = (
                word_ids
                .amax(dim=1)
                .add(1)
                .clamp_min(0)
            )

            max_words = max(
                1,
                int(
                    word_lengths
                    .max()
                    .item()
                ),
            )

            # Именно пространство ДО sequence_encoder
            # использовалось для semantic embeddings.
            semantic_word_hidden = (
                self.model._pool_words(
                    hidden,
                    word_ids,
                    subword_positions,
                    max_words=max_words,
                )
            )

            word_padding_mask = (
                torch.arange(
                    max_words,
                    device=self.device,
                )[None, :]
                >= word_lengths[:, None]
            )

            semantic_word_hidden = (
                semantic_word_hidden.masked_fill(
                    word_padding_mask[..., None],
                    0.0,
                )
            )

            # Реальная архитектура текущей модели.
            word_hidden = (
                self.model.sequence_encoder(
                    semantic_word_hidden,
                    src_key_padding_mask=word_padding_mask,
                )
            )

            logits = self.model.classifier(
                word_hidden
            )

        return (
            logits,
            semantic_word_hidden,
            word_lengths,
        )


    def _extract_runs(
        self,
        class_ids,
        word_lengths,
    ):
        """Объединяет соседние слова одинакового ненулевого класса в отдельные entities, сохраняя раздельными несмежные fragments одного класса."""

        class_ids = (
            class_ids
            .cpu()
            .numpy()
        )

        word_lengths = (
            word_lengths
            .cpu()
            .numpy()
        )

        runs = []

        for b, n_words in enumerate(word_lengths):
            ids = class_ids[
                b,
                :n_words,
            ]

            if not n_words:
                continue

            boundaries = np.flatnonzero(
                np.r_[
                    True,
                    ids[1:] != ids[:-1],
                    True,
                ]
            )

            for start, end in zip(
                boundaries[:-1],
                boundaries[1:],
            ):
                class_id = int(
                    ids[start]
                )

                if class_id != self.O_ID:
                    runs.append([
                        b,
                        class_id,
                        int(start),
                        int(end),
                    ])

        return runs


    def _rule_cleanup(
        self,
        texts,
        word_spans_batch,
        runs,
    ):
        """Удаляет заведомо невозможные class-word сочетания и самостоятельные служебные слова, при необходимости разделяя entity на несколько fragments."""

        cleaned = []

        for b, class_id, start, end in runs:
            text = texts[b]
            spans = word_spans_batch[b]

            segment_start = None

            # Sentinel False закрывает последний segment.
            for offset in range(
                end - start + 1
            ):
                if offset < end - start:
                    word_idx = start + offset
                    s, e = spans[word_idx]
                    word = text[s:e]

                    valid = not (
                        (
                            class_id
                            in self.NO_PURE_NUMBER
                            and word.isdigit()
                        )
                        or (
                            class_id
                            == self.ARTICLE_ID
                            and word.isalpha()
                        )
                    )
                else:
                    valid = False

                if valid:
                    if segment_start is None:
                        segment_start = (
                            start + offset
                        )

                    continue

                if segment_start is None:
                    continue

                segment_end = (
                    start + offset
                )

                # Entity = только "для", "с", "и"...
                if (
                    segment_end
                    - segment_start
                    == 1
                ):
                    s, e = spans[
                        segment_start
                    ]

                    if (
                        text[s:e].casefold()
                        in self.IMPOSSIBLE_STANDALONE
                    ):
                        segment_start = None
                        continue

                cleaned.append([
                    b,
                    class_id,
                    segment_start,
                    segment_end,
                ])

                segment_start = None

        return cleaned


    @torch.inference_mode()
    def _semantic_cleanup(
        self,
        semantic_word_hidden,
        runs,
    ):
        """Получает mean embedding каждой entity из уже рассчитанных word embeddings и применяет исходные cluster relabel/drop правила через cosine similarity."""

        if not runs:
            return []

        device = self.device

        run_array = np.asarray(
            runs,
            dtype=np.int64,
        )

        coords = torch.as_tensor(
            run_array,
            device=device,
            dtype=torch.long,
        )

        b_idx = coords[:, 0]
        predicted_ids = coords[:, 1]
        starts = coords[:, 2]
        ends = coords[:, 3]

        # Prefix sums позволяют получить
        # mean любого contiguous span за O(1).
        prefix = F.pad(
            semantic_word_hidden
            .float()
            .cumsum(dim=1),
            (0, 0, 1, 0),
        )

        entity_embeddings = (
            prefix[b_idx, ends]
            - prefix[b_idx, starts]
        )

        entity_embeddings /= (
            (ends - starts)
            .float()
            .unsqueeze(1)
        )

        entity_embeddings = F.normalize(
            entity_embeddings,
            p=2,
            dim=1,
        )

        n_entities = len(runs)
        n_classes = len(self.class_names)

        # Один matmul сразу до всех cluster centers.
        similarities = (
            entity_embeddings
            @ self.all_centers.T
        )
        
        scores = torch.full(
            (n_entities, n_classes),
            -torch.inf,
            device=device,
        )
        
        # Для каждого центра указываем его исходный semantic class.
        center_classes = (
            self.center_class_ids[None, :]
            .expand(n_entities, -1)
        )
        
        # Для каждого класса оставляем максимальный cosine
        # среди только его собственных cluster centers.
        scores.scatter_reduce_(
            dim=1,
            index=center_classes,
            src=similarities,
            reduce="amax",
            include_self=True,
        )



        

        own_score = scores.gather(
            1,
            predicted_ids[:, None],
        ).squeeze(1)

        best_score, best_id = (
            scores.max(dim=1)
        )

        top2 = scores.topk(
            k=min(
                2,
                scores.size(1),
            ),
            dim=1,
        ).values

        if top2.size(1) == 2:
            second_score = top2[:, 1]
        else:
            second_score = torch.full_like(
                best_score,
                -torch.inf,
            )

        gap = (
            best_score
            - own_score
        )

        dominance = (
            best_score
            - second_score
        )

        own_available = torch.isfinite(
            own_score
        )

        agrees = (
            best_id
            == predicted_ids
        )

        new_ids = predicted_ids.clone()

        keep = torch.ones(
            n_entities,
            dtype=torch.bool,
            device=device,
        )

        # Confident relabel.
        relabel = (
            own_available
            & ~agrees
            & self.relabel_target_mask[
                best_id
            ]
            & (
                best_score
                >= self.min_sim[best_id]
            )
            & (
                gap
                >= self.min_gap[best_id]
            )
            & (
                dominance
                >= self.RELABEL_MIN_DOMINANCE
            )
        )

        new_ids[relabel] = (
            best_id[relabel]
        )

        undecided = (
            own_available
            & ~agrees
            & ~relabel
        )

        # Type contradiction.
        type_reject = (
            undecided
            & (
                predicted_ids
                == self.TYPE_ID
            )
            & self.relabel_target_mask[
                best_id
            ]
            & (
                gap
                >= self.TYPE_REJECT_GAP
            )
        )

        keep[type_reject] = False

        # Strong general contradiction.
        general_reject = (
            undecided
            & ~type_reject
            & (
                gap
                >= self.GENERAL_REJECT_GAP
            )
        )

        keep[general_reject] = False

        # Weak own-class similarity.
        low_own_reject = (
            undecided
            & ~type_reject
            & ~general_reject
            & (
                own_score
                < self.LOW_OWN_SIM
            )
            & (
                gap
                >= self.LOW_OWN_REJECT_GAP
            )
        )

        keep[low_own_reject] = False

        keep = (
            keep
            .cpu()
            .numpy()
        )

        new_ids = (
            new_ids
            .cpu()
            .numpy()
        )

        result = []

        for run, should_keep, new_id in zip(
            runs,
            keep,
            new_ids,
        ):
            if should_keep:
                result.append([
                    run[0],
                    int(new_id),
                    run[2],
                    run[3],
                ])

        return result


    def _runs_to_class_ids(
        self,
        batch_size,
        max_words,
        runs,
    ):
        """Преобразует surviving semantic entities обратно в компактную word-level матрицу классов для последующего расширения spans."""

        class_ids = np.full(
            (batch_size, max_words),
            self.O_ID,
            dtype=np.int16,
        )

        for b, class_id, start, end in runs:
            class_ids[
                b,
                start:end,
            ] = class_id

        return class_ids


    def _expand_entities(
        self,
        texts,
        word_spans_batch,
        class_ids,
        word_lengths,
    ):
        """Расширяет entities правилами ADJ+NOUN и PREPOSITION+next, вызывая морфологию только для реально проверяемых слов."""
    
        word_lengths = word_lengths.cpu().numpy()
    
        for b, n_words in enumerate(word_lengths):
            if n_words < 2:
                continue
    
            text = texts[b]
            spans = word_spans_batch[b]
            ids = class_ids[b]
    
            words = [
                text[start:end].casefold()
                for start, end in spans[:n_words]
            ]
    
            while True:
                changed = False
    
                # 1. ADJ внутри entity + следующий NOUN.
                for i in range(n_words - 1):
                    class_id = ids[i]
    
                    if class_id == self.O_ID:
                        continue
    
                    if (
                        self._pos(words[i]) == "ADJF"
                        and self._pos(words[i + 1]) == "NOUN"
                        and ids[i + 1] != class_id
                    ):
                        ids[i + 1] = class_id
                        changed = True
    
                # 2. PREPOSITION внутри entity + следующее слово.
                # Морфология здесь вообще не вызывается.
                for i in range(n_words - 1):
                    class_id = ids[i]
    
                    if (
                        class_id != self.O_ID
                        and words[i] in self.PREPOSITIONS
                        and ids[i + 1] != class_id
                    ):
                        ids[i + 1] = class_id
                        changed = True
    
                if not changed:
                    break
    
        return class_ids


    def _final_attributes(
        self,
        texts,
        word_spans_batch,
        class_ids,
        word_lengths,
    ):
        """Собирает итог: contiguous fragments становятся entities, а несколько surviving entities одного класса объединяются через запятую в единственное значение."""

        result = []

        word_lengths = (
            word_lengths
            .cpu()
            .numpy()
        )

        for b, n_words in enumerate(word_lengths):
            ids = class_ids[
                b,
                :n_words,
            ]

            text = texts[b]
            spans = word_spans_batch[b]

            values = {}

            if n_words:
                boundaries = np.flatnonzero(
                    np.r_[
                        True,
                        ids[1:] != ids[:-1],
                        True,
                    ]
                )

                for start, end in zip(
                    boundaries[:-1],
                    boundaries[1:],
                ):
                    class_id = int(
                        ids[start]
                    )

                    if class_id == self.O_ID:
                        continue

                    label = self.id2label[
                        class_id
                    ]

                    value = text[
                        spans[start][0]:
                        spans[end - 1][1]
                    ]

                    values.setdefault(
                        label,
                        [],
                    ).append(value)

            result.append({
                label: ", ".join(items)
                for label, items
                in values.items()
            })

        return result

    def _prepare_batch(
        self,
        texts,
    ):
        """
        Токенизирует исходные строки без промежуточных NumPy-тензоров и
        точно восстанавливает прежние word_ids/subword_positions через offsets.
        """
    
        word_spans_batch = [
            self._get_word_spans(text)
            for text in texts
        ]
    
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
        )
    
        word_ids_batch = []
        subword_positions_batch = []
    
        for encoding, spans in zip(
            encoded.encodings,
            word_spans_batch,
        ):
            offsets = encoding.offsets
    
            word_ids = [-1] * len(offsets)
            subword_positions = [0] * len(offsets)
    
            word_idx = 0
            previous_word = -1
            subword_position = 0
    
            for token_idx, (start, end) in enumerate(offsets):
                # Special/padding token.
                if start == end:
                    continue
    
                # Ищем первый WORD_RE-span,
                # заканчивающийся после начала токена.
                while (
                    word_idx < len(spans)
                    and spans[word_idx][1] <= start
                ):
                    word_idx += 1
    
                if word_idx >= len(spans):
                    break
    
                word_start, word_end = spans[word_idx]
    
                # То же overlap-условие,
                # что использовалось в старом alignment.
                if (
                    start < word_end
                    and end > word_start
                ):
                    word_ids[token_idx] = word_idx
    
                    if word_idx == previous_word:
                        subword_position += 1
                    else:
                        previous_word = word_idx
                        subword_position = 0
    
                    subword_positions[
                        token_idx
                    ] = subword_position
    
            word_ids_batch.append(word_ids)
            subword_positions_batch.append(
                subword_positions
            )
    
        return (
            encoded,
            word_spans_batch,
            word_ids_batch,
            subword_positions_batch,
        )

    @torch.inference_mode()
    def _process_batch(
        self,
        texts,
    ):
        """
        Обрабатывает один GPU batch: быстрый Python preprocessing,
        NER forward, cleanup, expansion и финальную сборку attributes.
        """
    
        (
            encoded,
            word_spans_batch,
            word_ids,
            subword_positions,
        ) = self._prepare_batch(texts)
    
        # Сразу Python lists → GPU.
        # Большой промежуточный NumPy-массив больше не создаётся.
        input_ids = torch.tensor(
            encoded["input_ids"],
            device=self.device,
            dtype=torch.long,
        )
    
        attention_mask = torch.tensor(
            encoded["attention_mask"],
            device=self.device,
            dtype=torch.long,
        )
    
        word_ids = torch.tensor(
            word_ids,
            device=self.device,
            dtype=torch.long,
        )
    
        subword_positions = torch.tensor(
            subword_positions,
            device=self.device,
            dtype=torch.long,
        )
    
        token_type_ids = None
    
        if "token_type_ids" in encoded:
            token_type_ids = torch.tensor(
                encoded["token_type_ids"],
                device=self.device,
                dtype=torch.long,
            )
    
        (
            logits,
            semantic_word_hidden,
            word_lengths,
        ) = self._forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_ids=word_ids,
            subword_positions=subword_positions,
            token_type_ids=token_type_ids,
        )
    
        # 1. Argmax.
        class_ids = logits.argmax(
            dim=-1
        )
    
        # 2. Одинаковые соседние labels → entities.
        runs = self._extract_runs(
            class_ids,
            word_lengths,
        )
    
        # 3. Rule cleanup.
        runs = self._rule_cleanup(
            texts,
            word_spans_batch,
            runs,
        )
    
        # 4. Semantic cleanup/relabel.
        runs = self._semantic_cleanup(
            semantic_word_hidden,
            runs,
        )
    
        # 5. Entities → word-level classes.
        class_ids = self._runs_to_class_ids(
            batch_size=len(texts),
            max_words=logits.size(1),
            runs=runs,
        )
    
        # 6. ADJ+NOUN / PREPOSITION+next.
        class_ids = self._expand_entities(
            texts,
            word_spans_batch,
            class_ids,
            word_lengths,
        )
    
        # 7. Итоговые attributes.
        return self._final_attributes(
            texts,
            word_spans_batch,
            class_ids,
            word_lengths,
        )


    
        
    @torch.inference_mode()
    def _process_prepared_batch(
        self,
        texts,
        encoded,
        word_spans_batch,
        word_ids,
        subword_positions,
    ):
        """Обрабатывает уже токенизированный и выровненный batch через GPU NER и postprocessing."""
    
        input_ids = torch.as_tensor(
            encoded["input_ids"],
            device=self.device,
            dtype=torch.long,
        )
    
        attention_mask = torch.as_tensor(
            encoded["attention_mask"],
            device=self.device,
            dtype=torch.long,
        )
    
        word_ids = torch.as_tensor(
            word_ids,
            device=self.device,
            dtype=torch.long,
        )
    
        subword_positions = torch.as_tensor(
            subword_positions,
            device=self.device,
            dtype=torch.long,
        )
    
        token_type_ids = None
    
        if "token_type_ids" in encoded:
            token_type_ids = torch.as_tensor(
                encoded["token_type_ids"],
                device=self.device,
                dtype=torch.long,
            )
    
        (
            logits,
            semantic_word_hidden,
            word_lengths,
        ) = self._forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_ids=word_ids,
            subword_positions=subword_positions,
            token_type_ids=token_type_ids,
        )
    
        class_ids = logits.argmax(dim=-1)
    
        runs = self._extract_runs(
            class_ids,
            word_lengths,
        )
    
        runs = self._rule_cleanup(
            texts,
            word_spans_batch,
            runs,
        )
    
        runs = self._semantic_cleanup(
            semantic_word_hidden,
            runs,
        )
    
        class_ids = self._runs_to_class_ids(
            batch_size=len(texts),
            max_words=logits.size(1),
            runs=runs,
        )
    
        class_ids = self._expand_entities(
            texts,
            word_spans_batch,
            class_ids,
            word_lengths,
        )
    
        return self._final_attributes(
            texts,
            word_spans_batch,
            class_ids,
            word_lengths,
        )


    def predict_batch(
        self,
        texts,
        batch_size=6000,
    ):
        """
        Разбивает names на GPU batches и возвращает attributes
        в исходном порядке.
        """
    
        texts = list(texts)
    
        if len(texts) == 0:
            return []
    
        results = []
    
        for start in range(
            0,
            len(texts),
            batch_size,
        ):
            results.extend(
                self._process_batch(
                    texts[
                        start:
                        start + batch_size
                    ]
                )
            )
    
        return results


    def predict(self, text):
        """Обрабатывает одно название товара через тот же batch pipeline и возвращает один словарь извлечённых атрибутов."""

        return self._process_batch(
            [text]
        )[0]



## PhysicalAttributeParser

Алгоритмически извлекает физические характеристики непосредственно из названия товара.

Что делает:

- распознаёт числа и единицы измерения;
- извлекает массу, объём, длину, ширину, высоту и другие физические характеристики;
- обрабатывает многомерные размеры;
- защищает числовые конструкции, которые не являются физическими величинами;
- возвращает найденные характеристики как словарь атрибутов.

Не использует нейросеть.

In [5]:
import re


class PhysicalAttributeParser:
    """
    Алгоритмический парсер физических величин:
    размеры, масса, объём, мощность, память и другие значения с единицами измерения.
    """

    def __init__(self):
        """Создаёт словари единиц и один раз компилирует все regex, используемые при последующем парсинге."""

        # ============================================================
        # 1. DEFAULT ATTRIBUTE BY UNIT
        # ============================================================

        self.default_unit_attributes = {
            # Геометрия без явного указателя
            "нм": "длина",
            "мкм": "длина",
            "мм": "длина",
            "см": "длина",
            "дм": "длина",
            "м": "длина",
            "км": "длина",
            "дюйм": "длина",
            "ft": "длина",
        
            # Объём
            "мкл": "объем",
            "мл": "объем",
            "л": "объем",
            "мм³": "объем",
            "см³": "объем",
            "дм³": "объем",
            "м³": "объем",
        
            # Масса
            "мг": "вес",
            "г": "вес",
            "кг": "вес",
            "т": "вес",
        
            # Количество
            "шт": "количество",
        
            # Электрика
            "вт": "мощность",
            "квт": "мощность",
            "в": "напряжение",
            "кв": "напряжение",
            "а": "сила тока",
        
            # Данные
            "мб": "объем памяти",
            "гб": "объем памяти",
            "тб": "объем памяти",
        
            # Частота
            "гц": "частота",
            "кгц": "частота",
            "мгц": "частота",
            "ггц": "частота",
        
            # Время
            "мс": "время",
            "с": "время",
            "мин": "время",
            "ч": "время",
        
            "%": "процент",
        }



        # ============================================================
        # 2. POSSIBLE UNITS FOR EXPLICIT ATTRIBUTE
        # ============================================================




        self.keyword_possible_units = {
            # Геометрия
            "ширина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft"},
            "высота": {"нм", "мкм", "мм", "см", "дм", "м", "дюйм", "ft", "u"},
            "длина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft", "пог. м"},
            "глубина": {"мкм", "мм", "см", "дм", "м", "дюйм"},
            "толщина": {"нм", "мкм", "мм", "см", "м", "дюйм"},
            "диаметр": {"мкм", "мм", "см", "м", "дюйм", "ft"},
            "радиус": {"мм", "см", "м", "км", "дюйм"},
            "размер": {"мкм", "мм", "см", "м", "дюйм", "ft"},
            "габарит": {"мм", "см", "м", "дюйм"},
            "периметр": {"мм", "см", "м"},
            "окружность": {"мм", "см", "м"},
            "клиренс": {"мм", "см", "м"},
            "дорожный просвет": {"мм", "см", "м"},
            "колея": {"мм", "см", "м"},
            "база": {"мм", "см", "м"},
            "размах": {"мм", "см", "м"},
            "ход": {"мкм", "мм", "см", "м"},
            "шаг": {"мкм", "мм", "см", "м"},
            "зазор": {"мкм", "мм", "см", "м"},
            "угол": {"°", "град", "рад"},
        
            # Площадь
            "площадь": {
                "мм²", "см²", "дм²", "м²", "км²",
                "кв. мм", "кв. см", "кв. м",
                "га"
            },
        
            # Масса / механика
            "вес": {"мг", "г", "кг", "т", "lb", "oz", "карат"},
            "масса": {"мг", "г", "кг", "т", "lb", "oz"},
            "максимальный вес": {"г", "кг", "т", "lb"},
            "допустимый вес": {"г", "кг", "т", "lb"},
        
            "нагрузка": {"г", "кг", "т", "н", "кн", "кгс", "lb"},
            "грузоподъемность": {"кг", "т", "lb"},
            "грузоподъёмность": {"кг", "т", "lb"},
        
            "сила": {"н", "кн", "мн", "кгс"},
            "усилие": {"н", "кн", "мн", "кгс"},
            "крутящий момент": {"н·м", "нм", "кгс·м", "lb-ft"},
        
            # Объём
            "объем": {
                "мкл", "мл", "л",
                "мм³", "см³", "дм³", "м³",
                "куб. мм", "куб. см", "куб. м",
                "кб", "мб", "гб", "тб"
            },
            "объём": {
                "мкл", "мл", "л",
                "мм³", "см³", "дм³", "м³",
                "куб. мм", "куб. см", "куб. м",
                "кб", "мб", "гб", "тб"
            },
        
            "вместимость": {
                "мл", "л", "см³", "дм³", "м³",
                "шт", "бутылок", "листов", "комплектов", "чел"
            },
        
            "емкость": {
                "мкл", "мл", "л",
                "см³", "дм³", "м³",
                "мкф", "нф", "пф", "ф",
                "мач", "ач",
                "вт·ч", "квт·ч",
                "мб", "гб", "тб",
                "шт"
            },
            "ёмкость": {
                "мкл", "мл", "л",
                "см³", "дм³", "м³",
                "мкф", "нф", "пф", "ф",
                "мач", "ач",
                "вт·ч", "квт·ч",
                "мб", "гб", "тб",
                "шт"
            },
        
            "резервуар": {"мл", "л", "см³", "дм³", "м³"},
            "бак": {"мл", "л", "см³", "дм³", "м³"},
            "объем бака": {"мл", "л", "см³", "дм³", "м³"},
            "объём бака": {"мл", "л", "см³", "дм³", "м³"},
        
            # Время
            "время": {"мкс", "мс", "с", "сек", "мин", "ч", "сут", "дни", "месяцы"},
            "длительность": {"мс", "с", "сек", "мин", "ч", "сут"},
            "продолжительность": {"мс", "с", "сек", "мин", "ч", "сут", "месяцы"},
        
            # Скорость
            "скорость": {
                "мм/с", "см/с", "м/с", "м/мин", "км/ч",
                "об/мин",
                "кадр/с", "стр/мин", "стежки/мин",
                "кб/с", "мб/с", "гб/с",
                "кбит/с", "мбит/с", "гбит/с",
                "iops", "ips"
            },
        
            "скорость вращения": {"об/мин", "об/с", "rpm"},
            "обороты": {"об/мин", "об/с", "rpm"},
            "число оборотов": {"об/мин", "об/с", "rpm"},
        
            "скорость передачи": {
                "бит/с", "кбит/с", "мбит/с", "гбит/с",
                "байт/с", "кб/с", "мб/с", "гб/с"
            },
            "скорость чтения": {"кб/с", "мб/с", "гб/с", "iops"},
            "скорость записи": {"кб/с", "мб/с", "гб/с", "iops"},
            "пропускная способность": {
                "бит/с", "кбит/с", "мбит/с", "гбит/с",
                "байт/с", "кб/с", "мб/с", "гб/с"
            },
            "битрейт": {"бит/с", "кбит/с", "мбит/с", "гбит/с"},
        
            # Расход / производительность
            "расход": {
                "мл/мин", "мл/ч",
                "л/мин", "л/ч", "л/сут",
                "м³/мин", "м³/ч", "м³/сут",
                "г/мин", "г/ч",
                "кг/мин", "кг/ч", "кг/сут",
                "л/кг", "л/м²", "кг/м²"
            },
        
            "производительность": {
                "г/мин", "г/ч",
                "кг/мин", "кг/ч", "кг/сут",
                "мл/мин", "л/мин", "л/ч",
                "м³/мин", "м³/ч", "м³/сут",
                "шт/мин", "шт/ч"
            },
        
            "расход воздуха": {"л/мин", "л/с", "м³/мин", "м³/ч", "cfm"},
            "расход воды": {"мл/мин", "л/мин", "л/ч", "м³/ч"},
            "расход топлива": {"л/ч", "л/100км", "кг/ч"},
            "воздушный поток": {"л/с", "л/мин", "м³/мин", "м³/ч", "cfm"},
        
            # Электрика
            "мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с.", "btu/h"},
            "потребляемая мощность": {"мвт", "вт", "квт", "ва", "ква"},
            "выходная мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с."},
        
            "напряжение": {"мв", "в", "кв"},
            "ток": {"мка", "ма", "а", "ка"},
            "сила тока": {"мка", "ма", "а", "ка"},
            "сопротивление": {"мом", "ом", "ком", "мом"},
            "заряд": {"мккл", "мкл", "кл", "мач", "ач"},
        
            "электрическая емкость": {"пф", "нф", "мкф", "мф", "ф"},
            "электрическая ёмкость": {"пф", "нф", "мкф", "мф", "ф"},
        
            # Энергия
            "энергия": {"дж", "кдж", "мдж", "вт·ч", "квт·ч", "кал", "ккал"},
            "энергопотребление": {"вт·ч", "квт·ч", "квт·ч/год"},
        
            # Частота
            "частота": {"гц", "кгц", "мгц", "ггц"},
            "частота вращения": {"об/мин", "об/с", "гц"},
            "частота обновления": {"гц"},
            "частота кадров": {"кадр/с", "fps"},
        
            # Температура
            "температура": {"°c", "°с", "°f", "k"},
        
            # Давление
            "давление": {
                "па", "кпа", "мпа",
                "бар", "мбар", "атм", "psi",
                "мм рт. ст.", "кгс/см²"
            },
        
            # Память / данные
            "память": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "объем памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "объём памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "оперативная память": {"мб", "гб", "тб"},
            "накопитель": {"мб", "гб", "тб"},
        
            "разрядность": {"бит"},
            "битность": {"бит"},
        
            # Оптика / свет
            "световой поток": {"лм"},
            "освещенность": {"лк"},
            "освещённость": {"лк"},
            "яркость": {"кд/м²", "нит"},
        
            # Свойства вещества
            "плотность": {
                "мг/мл", "г/мл", "г/л",
                "г/см³", "кг/л", "кг/м³"
            },
            "вязкость": {"па·с", "мпа·с", "сст", "cst"},
            "твердость": {"shore a", "shore d", "hb", "hrc", "hv"},
            "твёрдость": {"shore a", "shore d", "hb", "hrc", "hv"},
        
            # Проценты / относительные характеристики
            "процент": {"%"},
            "доля": {"%"},
            "концентрация": {"%", "мг/мл", "мг/л", "г/л", "моль/л"},
            "влажность": {"%"},
        
            # Шум
            "уровень шума": {"дб", "дба"},
            "шум": {"дб", "дба"},
        }
        
        
        
        
        
        self.unit_aliases = {
            "мм": {
                "мм",
                "мм.",
                "миллиметр",
                "миллиметра",
                "миллиметров",
                "в мм",
                "в мм.",
                "в миллиметрах",
                "в милиметрах",
                "в миллииметрах",
            },
        
            "см": {
                "см",
                "см.",
                "сантиметр",
                "сантиметра",
                "сантиметров",
                "в см",
                "в сантиметрах",
                "в сантиметрх",
                "в сантиментрах",
                "в сантимерах",
            },
        
            "м": {
                "м",
                "метр",
                "метра",
                "метров",
                "в м",
                "в метрах",
            },
        
            "дюйм": {
                '"',
                "дюйм",
                "дюймы",
                "дюймов",
                "в дюймах",
                "(дюйм)",
            },
        
            "мл": {
                "мл",
                "мл.",
                "миллилитр",
                "миллилитров",
                "в мл",
                "в мл.",
                "в миллилитрах",
                "в милилитрах",
                "в миллиллитрах",
            },
        
            "л": {
                "л",
                "литр",
                "литров",
                "в л",
                "в литрах",
            },
        
            "г": {
                "г",
                "гр",
                "грамм",
                "граммов",
                "в г",
                "в гр",
                "в граммах",
            },
        
            "кг": {
                "кг",
                "килограмм",
                "килограммов",
                "в кг",
                "в килограммах",
                "в киллограммах",
                "в килогораммах",
            },
        
            "гб": {
                "гб",
                "гигабайт",
                "гигабайтов",
                "в гб",
                "в гигабайтах",
            },
        
            "мб": {
                "мб",
                "мегабайт",
                "мегабайтов",
                "в мб",
                "в мегабайтах",
            },
        
            "%": {
                "%",
                "процент",
                "процентов",
                "в %",
                "в процентах",
                "в массовых процентах",
            },
        }
        
        self.unit_aliases.update({
            "ч": {
                "ч",
                "час",
                "часа",
                "часов",
            },
            "мин": {
                "мин",
                "мин.",
                "минута",
                "минуты",
                "минут",
            },
            "с": {
                "с",
                "сек",
                "сек.",
                "секунда",
                "секунды",
                "секунд",
            },
            "вт": {
                "вт",
                "ватт",
                "ватта",
                "ваттов",
                "w",
            },
            "квт": {
                "квт",
                "киловатт",
                "киловатта",
                "киловаттов",
                "kw",
            },
        })
        
        extra_unit_aliases = {
            # Геометрия
            "нм": {
                "нм", "нм.",
                "нанометр", "нанометра", "нанометров",
                "nanometer", "nanometers", "nm",
            },
            "мкм": {
                "мкм", "мкм.",
                "микрометр", "микрометра", "микрометров",
                "микрон", "микрона", "микронов",
                "um", "µm", "μm",
            },
            "дм": {
                "дм", "дм.",
                "дециметр", "дециметра", "дециметров",
                "decimeter", "decimeters", "dm",
            },
            "км": {
                "км", "км.",
                "километр", "километра", "километров",
                "kilometer", "kilometers", "km",
            },
            "ft": {
                "ft", "foot", "feet",
                "фут", "фута", "футов",
            },
            "пог. м": {
                "пог. м", "пог.м", "пог м",
                "погонный метр", "погонного метра", "погонных метров",
            },
        
            # Угол
            "°": {
                "°",
                "градус", "градуса", "градусов",
                "deg", "degree", "degrees",
            },
            "рад": {
                "рад", "рад.",
                "радиан", "радиана", "радианов",
                "radian", "radians",
            },
        
            # Площадь
            "мм²": {
                "мм²", "мм2", "mm²", "mm2",
                "кв. мм", "кв.мм", "кв мм",
            },
            "см²": {
                "см²", "см2", "cm²", "cm2",
                "кв. см", "кв.см", "кв см",
            },
            "дм²": {
                "дм²", "дм2", "dm²", "dm2",
                "кв. дм", "кв.дм", "кв дм",
            },
            "м²": {
                "м²", "м2", "m²", "m2",
                "кв. м", "кв.м", "кв м",
            },
            "км²": {
                "км²", "км2", "km²", "km2",
                "кв. км", "кв.км", "кв км",
            },
            "га": {
                "га", "гектар", "гектара", "гектаров",
                "hectare", "hectares",
            },
        
            # Объём
            "мкл": {
                "мкл", "мкл.",
                "микролитр", "микролитра", "микролитров",
                "µl", "μl", "ul",
            },
            "мм³": {
                "мм³", "мм3", "mm³", "mm3",
                "куб. мм", "куб.мм", "куб мм",
            },
            "см³": {
                "см³", "см3", "cm³", "cm3",
                "куб. см", "куб.см", "куб см",
                "cc", "ccm",
            },
            "дм³": {
                "дм³", "дм3", "dm³", "dm3",
                "куб. дм", "куб.дм", "куб дм",
            },
            "м³": {
                "м³", "м3", "m³", "m3",
                "куб. м", "куб.м", "куб м",
            },
        
            # Масса
            "мг": {
                "мг", "мг.",
                "миллиграмм", "миллиграмма", "миллиграммов",
                "mg",
            },
            "т": {
                "т", "т.",
                "тонна", "тонны", "тонн",
                "ton", "tons", "tonne", "tonnes",
            },
            "lb": {
                "lb", "lbs",
                "фунт", "фунта", "фунтов",
                "pound", "pounds",
            },
            "oz": {
                "oz",
                "унция", "унции", "унций",
                "ounce", "ounces",
            },
            "карат": {
                "карат", "карата", "каратов",
                "ct", "carat", "carats",
            },
        
            # Сила / механика
            "н": {
                "н", "н.",
                "ньютон", "ньютона", "ньютонов",
                "newton", "newtons",
            },
            "кн": {
                "кн", "кн.",
                "килоньютон", "килоньютона", "килоньютонов",
                "kn",
            },
            "мн": {
                "мн",
                "меганьютон", "меганьютона", "меганьютонов",
            },
            "кгс": {
                "кгс", "kgf",
                "килограмм-сила", "килограмм силы",
            },
            "н·м": {
                "н·м", "н-м", "н*м",
                "n·m", "n-m", "n*m",
                "ньютон-метр",
            },
            "кгс·м": {
                "кгс·м", "кгс м", "кгс-м",
                "kgf·m", "kgf m",
            },
            "lb-ft": {
                "lb-ft", "lb ft", "lb·ft",
                "ft-lb", "ft lb",
            },
        
            # Время
            "мкс": {
                "мкс", "µs", "μs", "us",
                "микросекунда", "микросекунды", "микросекунд",
            },
            "мс": {
                "мс", "ms",
                "миллисекунда", "миллисекунды", "миллисекунд",
            },
            "сут": {
                "сут", "сут.",
                "сутки", "суток",
                "день", "дня", "дней",
                "day", "days",
            },
            "месяцы": {
                "месяц", "месяца", "месяцев", "месяцы",
                "month", "months",
            },
        
            # Скорость
            "мм/с": {"мм/с", "мм/сек", "mm/s", "mm/sec"},
            "см/с": {"см/с", "см/сек", "cm/s", "cm/sec"},
            "м/с": {"м/с", "м/сек", "m/s", "m/sec"},
            "м/мин": {"м/мин", "m/min"},
            "км/ч": {"км/ч", "км/час", "кмч", "km/h", "kmh", "kph"},
            "об/мин": {
                "об/мин", "об.мин", "об./мин",
                "оборот/мин", "оборотов/мин",
                "rpm", "r/min",
            },
            "об/с": {"об/с", "об/сек", "оборотов/с", "rps"},
            "кадр/с": {
                "кадр/с", "кадров/с",
                "кадр/сек", "кадров/сек",
                "fps",
            },
        
            # Передача данных
            "бит/с": {"бит/с", "бит/сек", "bps", "bit/s"},
            "кбит/с": {"кбит/с", "кбит/сек", "kbps", "kbit/s"},
            "мбит/с": {"мбит/с", "мбит/сек", "mbit/s", "mbps"},
            "гбит/с": {"гбит/с", "гбит/сек", "gbit/s", "gbps"},
            "кб/с": {"кб/с", "кб/сек", "kb/s", "kbyte/s"},
            "мб/с": {"мб/с", "мб/сек", "mb/s", "mbyte/s"},
            "гб/с": {"гб/с", "гб/сек", "gb/s", "gbyte/s"},
            "iops": {"iops"},
        
            # Расход
            "мл/мин": {"мл/мин", "ml/min"},
            "мл/ч": {"мл/ч", "ml/h", "ml/hr"},
            "л/мин": {"л/мин", "л/мин.", "l/min", "lpm"},
            "л/с": {"л/с", "л/сек", "l/s", "l/sec"},
            "л/ч": {"л/ч", "l/h", "l/hr"},
            "л/сут": {"л/сут", "л/сутки", "l/day"},
            "л/100км": {"л/100км", "л/100 км", "l/100km", "l/100 km"},
            "м³/мин": {"м³/мин", "м3/мин", "m³/min", "m3/min"},
            "м³/ч": {"м³/ч", "м3/ч", "m³/h", "m3/h"},
        
            # Количество
            "шт": {
                "шт", "шт.",
                "штук", "штука", "штуки",
                "ед", "ед.", "единица", "единицы", "единиц",
                "pcs", "pc",
            },
        
            # Мощность
            "мвт": {
                "мвт",
                "милливатт", "милливатта", "милливаттов",
            },
            "ва": {"ва", "va", "вольт-ампер", "вольт ампер"},
            "ква": {"ква", "kva", "киловольт-ампер", "киловольт ампер"},
            "л.с.": {
                "л.с.", "лс", "л. с.",
                "лошадиная сила", "лошадиных сил",
                "hp", "ps",
            },
        
            # Напряжение / ток
            "мв": {"мв", "mv", "милливольт", "милливольта", "милливольтов"},
            "в": {"в", "в.", "v", "вольт", "вольта", "вольтов"},
            "кв": {"кв", "kv", "киловольт", "киловольта", "киловольтов"},
            "мка": {"мка", "µa", "μa", "ua", "микроампер", "микроампера", "микроамперов"},
            "ма": {"ма", "ma", "миллиампер", "миллиампера", "миллиамперов"},
            "а": {"а", "а.", "a", "ампер", "ампера", "амперов"},
            "ка": {"ка", "ka", "килоампер", "килоампера", "килоамперов"},
        
            # Сопротивление
            "ом": {"ом", "ом.", "ohm", "Ω"},
            "ком": {"ком", "kohm", "kΩ", "килоом", "килоома", "килоомов"},
            "мом": {"мом", "mohm", "mΩ", "мегаом", "мегаома", "мегаомов"},
        
            # Ёмкость аккумулятора
            "мач": {
                "мач", "ма·ч", "ма*ч", "ма-ч",
                "mah", "ma·h", "ma-h",
                "миллиампер-час",
            },
            "ач": {
                "ач", "а·ч", "а*ч", "а-ч",
                "ah", "a·h", "a-h",
                "ампер-час",
            },
        
            # Электрическая ёмкость
            "пф": {"пф", "pf", "пикофарад", "пикофарада"},
            "нф": {"нф", "nf", "нанофарад", "нанофарада"},
            "мкф": {"мкф", "µf", "μf", "uf", "микрофарад", "микрофарада"},
            "мф": {"мф", "mf", "миллифарад", "миллифарада"},
            "ф": {"ф", "f", "фарад", "фарада"},
        
            # Энергия
            "дж": {"дж", "j", "джоуль", "джоуля", "джоулей"},
            "кдж": {"кдж", "kj", "килоджоуль", "килоджоуля", "килоджоулей"},
            "мдж": {"мдж", "mj", "мегаджоуль", "мегаджоуля", "мегаджоулей"},
            "вт·ч": {"вт·ч", "втч", "вт-ч", "вт*ч", "wh", "w·h", "w-h"},
            "квт·ч": {"квт·ч", "квтч", "квт-ч", "квт*ч", "kwh", "kw·h", "kw-h"},
        
            # Частота
            "гц": {"гц", "hz", "герц", "герца"},
            "кгц": {"кгц", "khz", "килогерц", "килогерца"},
            "мгц": {"мгц", "mhz", "мегагерц", "мегагерца"},
            "ггц": {"ггц", "ghz", "гигагерц", "гигагерца"},
        
            # Температура
            "°c": {
                "°c", "°с", "c°", "с°",
                "градус c", "градусов c",
                "градус цельсия", "градуса цельсия", "градусов цельсия",
                "celsius",
            },
            "°f": {
                "°f", "f°",
                "градус фаренгейта", "градусов фаренгейта",
                "fahrenheit",
            },
        
            # Давление
            "па": {"па", "pa", "паскаль", "паскаля", "паскалей"},
            "кпа": {"кпа", "kpa", "килопаскаль", "килопаскаля", "килопаскалей"},
            "мпа": {"мпа", "mpa", "мегапаскаль", "мегапаскаля", "мегапаскалей"},
            "бар": {"бар", "bar", "бара", "баров"},
            "мбар": {"мбар", "mbar", "миллибар", "миллибара", "миллибаров"},
            "атм": {"атм", "атм.", "атмосфера", "атмосферы", "атмосфер", "atm"},
            "psi": {"psi", "пси"},
        
            # Память
            "бит": {"бит", "бита", "битов", "bit", "bits"},
            "байт": {"байт", "байта", "байтов", "byte", "bytes"},
            "кб": {"кб", "kb", "kbyte", "килобайт", "килобайта", "килобайтов"},
            "тб": {"тб", "tb", "терабайт", "терабайта", "терабайтов"},
        
            # Свет
            "лм": {"лм", "lm", "люмен", "люмена", "люменов"},
            "лк": {"лк", "lx", "люкс", "люкса", "люксов"},
            "кд/м²": {"кд/м²", "кд/м2", "cd/m²", "cd/m2"},
            "нит": {"нит", "нита", "нитов", "nit", "nits"},
        
            # Шум
            "дб": {"дб", "db", "децибел", "децибела", "децибелов"},
            "дба": {"дба", "dba", "db(a)"},
        }


        for unit, aliases in extra_unit_aliases.items():
            self.unit_aliases.setdefault(
                unit,
                set(),
            ).update(aliases)


        # ============================================================
        # 4. GEOMETRY CONFIG
        # ============================================================

        self.dimension_keywords = (
            "дорожный просвет",
            "диаметр",
            "толщина",
            "ширина",
            "высота",
            "глубина",
            "длина",
            "радиус",
            "размер",
            "габарит",
            "клиренс",
            "колея",
            "база",
            "размах",
            "ход",
            "шаг",
            "зазор",
        )

        self.dimension_units = {
            "нм",
            "мкм",
            "мм",
            "см",
            "дм",
            "м",
            "км",
            "дюйм",
            "ft",
        }

        self.SEPARATORS = ",;/|()"


        # ============================================================
        # 5. BUILD INTERNAL STRUCTURES
        # ============================================================

        self._build_aliases()
        self._compile_patterns()


    def _build_aliases(self):
        """Строит быстрый lookup alias → canonical unit из всех канонических единиц и их допустимых вариантов написания."""

        all_units = (
            self.default_unit_attributes.keys()
            | self.unit_aliases.keys()
        )

        self.physical_unit_aliases = {
            alias.casefold(): canonical_unit
            for canonical_unit in all_units
            for alias in (
                self.unit_aliases.get(
                    canonical_unit,
                    set(),
                )
                | {canonical_unit}
            )
        }


    def _compile_patterns(self):
        """Один раз строит и компилирует regex для обычных величин, многомерных размеров, геометрических keywords и сокращённого диаметра."""

        dimension_unit_pattern = "|".join(
            map(
                re.escape,
                sorted(
                    (
                        alias
                        for alias, unit
                        in self.physical_unit_aliases.items()
                        if unit in self.dimension_units
                    ),
                    key=len,
                    reverse=True,
                ),
            )
        )

        self.multidimensional_pattern = re.compile(
            rf"""
            (?P<a>\d+(?:[.,]\d+)?)
            \s*[xх×*]\s*
            (?P<b>\d+(?:[.,]\d+)?)
            (?:
                \s*[xх×*]\s*
                (?P<c>\d+(?:[.,]\d+)?)
            )?
            \s*
            (?P<unit>{dimension_unit_pattern})
            (?![a-zа-яё0-9])
            """,
            re.VERBOSE,
        )

        unit_pattern = "|".join(
            map(
                re.escape,
                sorted(
                    self.physical_unit_aliases,
                    key=len,
                    reverse=True,
                ),
            )
        )

        self.physical_pattern = re.compile(
            rf"""
            (?P<number>\d+(?:[.,]\d+)?)
            \s*
            (?P<unit>{unit_pattern})
            (?![a-zа-яё0-9])
            """,
            re.VERBOSE,
        )

        self.dimension_keyword_pattern = re.compile(
            r"(?<!\w)("
            + "|".join(
                map(
                    re.escape,
                    sorted(
                        self.dimension_keywords,
                        key=len,
                        reverse=True,
                    ),
                )
            )
            + r")(?!\w)"
        )

        self.diameter_short_pattern = re.compile(
            r"(?:\bdia\s*|\bd\s*|[ø⌀]\s*)$"
        )


    def _get_dimension_attribute(
        self,
        name,
        number_start,
    ):
        """Определяет конкретный геометрический атрибут по локальному контексту перед числом; без подходящего keyword возвращает длину."""

        left_context = name[
            max(
                0,
                number_start - 40,
            ):
            number_start
        ]

        separator_pos = max(
            (
                left_context.rfind(char)
                for char in self.SEPARATORS
            ),
            default=-1,
        )

        local_context = left_context[
            separator_pos + 1:
        ]

        last_keyword = None

        for match in (
            self.dimension_keyword_pattern.finditer(
                local_context
            )
        ):
            last_keyword = match.group(1)

        if last_keyword is not None:
            return last_keyword

        if self.diameter_short_pattern.search(
            local_context
        ):
            return "диаметр"

        return "длина"


    def _get_physical_attribute(
        self,
        name,
        match,
        canonical_unit,
    ):
        """Определяет имя физического атрибута по единице измерения и контексту; геометрические единицы дополнительно анализируют текст слева."""

        if canonical_unit in self.dimension_units:
            return self._get_dimension_attribute(
                name,
                match.start("number"),
            )

        return self.default_unit_attributes.get(
            canonical_unit
        )


    def parse(self, name):
        """Извлекает из одного названия товара физические атрибуты, включая обычные величины и конструкции AxB/AxBxC."""

        name = str(name).casefold()

        result = {}

        # --------------------------------------------
        # AxB / AxBxC
        # --------------------------------------------

        multidimensional_spans = []

        for match in (
            self.multidimensional_pattern.finditer(
                name
            )
        ):
            canonical_unit = (
                self.physical_unit_aliases[
                    match.group("unit")
                ]
            )

            multidimensional_spans.append(
                match.span()
            )

            for attribute, value in zip(
                (
                    "длина",
                    "ширина",
                    "высота",
                ),
                match.group(
                    "a",
                    "b",
                    "c",
                ),
            ):
                if value is not None:
                    result.setdefault(
                        f"{attribute}, {canonical_unit}",
                        value.replace(
                            ",",
                            ".",
                        ),
                    )


        # --------------------------------------------
        # Обычные physical values
        # --------------------------------------------

        for match in (
            self.physical_pattern.finditer(
                name
            )
        ):
            # Не повторяем часть AxB / AxBxC.
            if any(
                start <= match.start() < end
                for start, end
                in multidimensional_spans
            ):
                continue

            raw_number = match.group(
                "number"
            )

            number = raw_number.replace(
                ",",
                ".",
            )

            alias = match.group(
                "unit"
            )

            canonical_unit = (
                self.physical_unit_aliases[
                    alias
                ]
            )

            # Не принимаем "2025г" за массу.
            if (
                canonical_unit == "г"
                and (
                    match.end("number")
                    == match.start("unit")
                )
                and raw_number.isdigit()
                and len(raw_number) == 4
                and (
                    1900
                    <= int(raw_number)
                    <= 2100
                )
            ):
                continue

            attribute = (
                self._get_physical_attribute(
                    name,
                    match,
                    canonical_unit,
                )
            )

            if attribute is not None:
                result.setdefault(
                    f"{attribute}, {canonical_unit}",
                    number,
                )

        return result


    def parse_batch(self, names):
        """Парсит последовательность названий товаров и возвращает список словарей физических атрибутов в том же порядке."""

        return [
            self.parse(name)
            for name in names
        ]

## ParallelPhysicalAttributeParser

Параллельная CPU-обёртка над `PhysicalAttributeParser`.

Что делает:

- разбивает большой список названий на chunks;
- распределяет их между несколькими процессами;
- создаёт `PhysicalAttributeParser` один раз на worker и переиспользует его;
- объединяет результаты с сохранением исходного порядка.

Используется для ускорения массового инференса.

In [6]:
from joblib import Parallel, delayed


_worker_physical_parser = None


def _get_worker_physical_parser():
    """Лениво создаёт один PhysicalAttributeParser внутри конкретного worker-процесса и затем переиспользует его."""

    global _worker_physical_parser

    if _worker_physical_parser is None:
        _worker_physical_parser = PhysicalAttributeParser()

    return _worker_physical_parser


def _parse_physical_chunk_worker(chunk):
    """Парсит один chunk через сохранённый PhysicalAttributeParser текущего worker-процесса."""

    parser = _get_worker_physical_parser()
    return parser.parse_batch(chunk)

class ParallelPhysicalAttributeParser:
    """
    Параллельная CPU-обёртка над PhysicalAttributeParser.
    Каждый worker создаёт parser один раз и переиспользует его.
    """

    def __init__(
        self,
        n_jobs=2,
        chunk_size=10_000,
    ):
        """Сохраняет число worker-процессов и размер чанка."""

        self.n_jobs = n_jobs
        self.chunk_size = chunk_size

        # Для одиночного parse multiprocessing не нужен.
        self._single_parser = PhysicalAttributeParser()


    def parse(self, text):
        """Парсит одно название напрямую без multiprocessing."""

        return self._single_parser.parse(text)


    def parse_batch(self, texts):
        """Разбивает names на чанки и параллельно парсит их через persistent workers."""

        texts = list(texts)

        if len(texts) == 0:
            return []

        if (
            self.n_jobs == 1
            or len(texts) <= self.chunk_size
        ):
            return self._single_parser.parse_batch(texts)

        chunks = [
            texts[i:i + self.chunk_size]
            for i in range(
                0,
                len(texts),
                self.chunk_size,
            )
        ]

        parts = Parallel(
            n_jobs=self.n_jobs,
            backend="loky",
            pre_dispatch=self.n_jobs,
            batch_size=1,
        )(
            delayed(
                _parse_physical_chunk_worker
            )(chunk)
            for chunk in chunks
        )

        return [
            item
            for part in parts
            for item in part
        ]

## ProductNameParser

Объединяет нейросетевой и алгоритмический парсинг названия товара.

Что делает:

- через `NerPostprocessor` извлекает смысловые атрибуты;
- через `PhysicalAttributeParser` извлекает физические характеристики;
- объединяет результаты двух парсеров в один словарь.

Это основной внешний интерфейс для парсинга названия товара.

Поддерживает обработку одной карточки и батча.

In [7]:
from joblib import Parallel, delayed

class ProductNameParser:
    """
    Объединяет NER-парсер и парсер физических величин
    в единый интерфейс.
    """

    def __init__(
        self,
        ner_postprocessor,
        physical_parser,
    ):
        """Сохраняет готовые NER и physical parser."""

        self.ner = ner_postprocessor
        self.physical = physical_parser


    def parse(self, text):
        """Разбирает одно название обоими парсерами."""

        ner_attributes = self.ner.predict(text)
        physical_attributes = self.physical.parse(text)

        return {
            **ner_attributes,
            **physical_attributes,
        }


    def parse_batch(
        self,
        texts,
        batch_size=6000,
    ):
        """Запускает NER и parallel physical parser."""
    
        texts = list(texts)
    
        ner_results = self.ner.predict_batch(
            texts,
            batch_size=batch_size,
        )
    
        physical_results = (
            self.physical.parse_batch(texts)
        )
    
        return [
            {
                **ner,
                **physical,
            }
            for ner, physical in zip(
                ner_results,
                physical_results,
            )
        ]

## Код работы парсера

In [8]:
"""
import torch

# from your_parser_module import (
#     WordNERModel,
#     NerPostprocessor,
#     PhysicalAttributeParser,
#     ProductNameParser,
# )

from transformers import AutoTokenizer

cards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human_normalized.parquet")

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 1000)

# ============================================================
# PATHS / CONFIG
# ============================================================

MODEL_DIR = (
    "/kaggle/input/datasets/kehhill/"
    "rubert-v3-ner/rubert_tiny2_word_ner"
)

CLUSTER_CENTERS_PATH = (
    "/kaggle/input/datasets/kehhill/"
    "ozon-e-cup/cluster_centers.pt"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)



CLASS_NAMES = (
    "O",
    "бренд",
    "тип",
    "модель",
    "материал",
    "цвет",
    "размер",
    "назначение",
    "артикул",
)

label2id = {
    label: i
    for i, label in enumerate(CLASS_NAMES)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

# КЛЮЧЕВЫЕ ПАРАМЕТРЫ
MAX_LENGTH = 100


# ============================================================
# LOAD NER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    use_fast=True,
)

model = WordNERModel.from_pretrained_dir(
    MODEL_DIR,
    num_classes=len(CLASS_NAMES),
    device=DEVICE,
)

# ============================================================
# CREATE PARSERS
# ============================================================

ner_postprocessor = NerPostprocessor(
    model=model,
    tokenizer=tokenizer,
    cluster_centers_path=CLUSTER_CENTERS_PATH,
    label2id=label2id,
    id2label=id2label,
    class_names=CLASS_NAMES,
    device=DEVICE,
    max_length=MAX_LENGTH,
    use_amp=True,
)


# ПРИ ИНФЕРЕНСЕ У НАС БУДЕТ 20 ЯДЕР КПУ, ПОЭТОМУ N_JOBS МОЖНО УВЕЛИЧИТЬ
physical_parser = ParallelPhysicalAttributeParser(
    n_jobs=2,
    chunk_size=10_000,
)

parser = ProductNameParser(
    ner_postprocessor=ner_postprocessor,
    physical_parser=physical_parser,
)


# ============================================================
# BATCH
# ============================================================

texts = [
    "weissgauff газовая варочная панель 60 см черная",
    "угловой диван лига диванов хьюго левый угол",
    "ssd samsung 1 тб",
]

# БАТЧ САЙЗ НАСТРАИВАЕТСЯ ЗДЕСЬ. БАТЧ САЙЗ 6К ЗАНИМАЕТ 13.6 ГБ GPU ПАМЯТИ, НА H100 ЛЕГКО ПОЙДЁТ И 20К
results = parser.parse_batch(
    texts,
    batch_size=6000,
)

print(results)

# ПРИ BATCH_SIZE=6к, N_JOBS=2 ДЛЯ АЛГО ПАРСЕРА
# СКОРОСТЬ 5 МИНУТ ДЛЯ 700К КАРТОЧЕК
# ОСНОВНЫЕ РАСХОДЫ: NER (40% ВРЕМЕНИ), АЛГО ПАРСЕР (10% ВРЕМЕНИ), ТОКЕНИЗАЦИЯ (25% ВРЕМЕНИ)
# ПРИ BATCH_SIZE=20к, N_JOBS=8 ДЛЯ АЛГО ПАРСЕРА ОЦЕНОЧНОЕ ВРЕМЯ ОКОЛО 2 МИНУТ

"""

'\nimport torch\n\n# from your_parser_module import (\n#     WordNERModel,\n#     NerPostprocessor,\n#     PhysicalAttributeParser,\n#     ProductNameParser,\n# )\n\nfrom transformers import AutoTokenizer\n\ncards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human_normalized.parquet")\n\npd.set_option(\'display.max_rows\', 100)\npd.set_option(\'display.max_colwidth\', 1000)\n\n# ============================================================\n# PATHS / CONFIG\n# ============================================================\n\nMODEL_DIR = (\n    "/kaggle/input/datasets/kehhill/"\n    "rubert-v3-ner/rubert_tiny2_word_ner"\n)\n\nCLUSTER_CENTERS_PATH = (\n    "/kaggle/input/datasets/kehhill/"\n    "ozon-e-cup/cluster_centers.pt"\n)\n\nDEVICE = torch.device(\n    "cuda" if torch.cuda.is_available() else "cpu"\n)\n\n\n\nCLASS_NAMES = (\n    "O",\n    "бренд",\n    "тип",\n    "модель",\n    "материал",\n    "цвет",\n    "размер",\n    "назначение",\n    "артикул",\n)\n\nlab

---

---

## Нормализация атрибутов товарных карточек

### Что делает код

Код приводит `attributes` карточек к единому виду, чтобы одинаковые характеристики не отличались только способом записи.

Основные виды нормализации:

- единицы измерения;
- числовые значения физических величин;
- многомерные размеры;
- синонимичные названия атрибутов.

### Общий пример

`"ширина": "12 см"` → `"ширина, мм": "120"`

`"число ядер": "8"` → `"количество ядер": "8"`


### Физические величины и единицы измерения

Единица измерения определяется только в достаточно надёжных случаях.

`длина` + `1.5 м` → `длина, мм: 1500`

`ширина, см` + `25` → `ширина, мм: 250`

`мощность (кВт)` + `1.2` → `мощность, Вт: 1200`


### Стандартизация числовых величин

Разные единицы одной физической величины переводятся в общий формат.

`1.5 кг` → `1500 г`

`2.3 м` → `2300 мм`

`0.75 л` → `750 мл`


### Многомерные размеры

`"размер упаковки": "120×80×50 мм"`

↓

`"длина упаковки": "120 мм"`  
`"ширина упаковки": "80 мм"`  
`"высота упаковки": "50 мм"`


### Стандартизация названий атрибутов

`число ядер` → `количество ядер`

`страна изготовителя` → `страна производителя`

`масса упаковки` → `вес упаковки`

`тип сенсора` → `тип датчика`


### Итоговый пайплайн

1. Исходные `attributes`
2. Определение физических величин и единиц измерения
3. Стандартизация единиц и числовых значений
4. Разбор многомерных размеров
5. Нормализация синонимичных названий атрибутов
6. Получение нормализованных `attributes`


## Стандартизирую и нормализую одномерные аттрибуты с физическими величинами и численными характеристиками

### Паттерн 1: keyword + подходящая единица измерения

Например:

`вес товара` + `2 кг` → `вес товара, г: 2000`

`длина кабеля` + `1.5 м` → `длина кабеля, мм: 1500`

`объем бака` + `3 л` → `объем бака, мл: 3000`

### Паттерн 2: единица измерения явно указана в названии атрибута

Например:

`ширина, см` + `25` → `ширина, мм: 250`

`мощность (кВт)` + `1.2` → `мощность, Вт: 1200`


## PhysicalUnitNormalizer

Нормализует физические величины в атрибутах товарных карточек.

Что делает:

- распознаёт единицы измерения в названии атрибута и его значении;
- приводит разные варианты записи единиц к каноническим;
- переводит величины в стандартные единицы;
- разбирает многомерные размеры вида `A×B` и `A×B×C`.

Примеры:

`"вес товара": "2 кг"` → `"вес товара, кг": "2"`

`"ширина, см": "25"` → `"ширина, мм": "250"`

`"размер": "120x60x75 см"` →
`длина, мм`, `ширина, мм`, `высота, мм`.

In [9]:
import re
from pint import UnitRegistry
import pandas as pd
import json
import re

class PhysicalUnitNormalizer:
    """
    Нормализует единицы измерения, численные физические величины
    и многомерные размеры в attributes товарных карточек.
    """

    def __init__(self):
        """Создаёт словари, regex и карту преобразований единиц один раз."""

        # ========================================================
        # CONFIG
        # ========================================================

        self.keyword_possible_units = {
            # Геометрия
            "ширина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft"},
            "высота": {"нм", "мкм", "мм", "см", "дм", "м", "дюйм", "ft", "u"},
            "длина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft", "пог. м"},
            "глубина": {"мкм", "мм", "см", "дм", "м", "дюйм"},
            "толщина": {"нм", "мкм", "мм", "см", "м", "дюйм"},
            "диаметр": {"мкм", "мм", "см", "м", "дюйм", "ft"},
            "радиус": {"мм", "см", "м", "км", "дюйм"},
            "размер": {"мкм", "мм", "см", "м", "дюйм", "ft"},
            "габарит": {"мм", "см", "м", "дюйм"},
            "периметр": {"мм", "см", "м"},
            "окружность": {"мм", "см", "м"},
            "клиренс": {"мм", "см", "м"},
            "дорожный просвет": {"мм", "см", "м"},
            "колея": {"мм", "см", "м"},
            "база": {"мм", "см", "м"},
            "размах": {"мм", "см", "м"},
            "ход": {"мкм", "мм", "см", "м"},
            "шаг": {"мкм", "мм", "см", "м"},
            "зазор": {"мкм", "мм", "см", "м"},
            "угол": {"°", "град", "рад"},
        
            # Площадь
            "площадь": {
                "мм²", "см²", "дм²", "м²", "км²",
                "кв. мм", "кв. см", "кв. м",
                "га"
            },
        
            # Масса / механика
            "вес": {"мг", "г", "кг", "т", "lb", "oz", "карат"},
            "масса": {"мг", "г", "кг", "т", "lb", "oz"},
            "максимальный вес": {"г", "кг", "т", "lb"},
            "допустимый вес": {"г", "кг", "т", "lb"},
        
            "нагрузка": {"г", "кг", "т", "н", "кн", "кгс", "lb"},
            "грузоподъемность": {"кг", "т", "lb"},
            "грузоподъёмность": {"кг", "т", "lb"},
        
            "сила": {"н", "кн", "мн", "кгс"},
            "усилие": {"н", "кн", "мн", "кгс"},
            "крутящий момент": {"н·м", "нм", "кгс·м", "lb-ft"},
        
            # Объём
            "объем": {
                "мкл", "мл", "л",
                "мм³", "см³", "дм³", "м³",
                "куб. мм", "куб. см", "куб. м",
                "кб", "мб", "гб", "тб"
            },
            "объём": {
                "мкл", "мл", "л",
                "мм³", "см³", "дм³", "м³",
                "куб. мм", "куб. см", "куб. м",
                "кб", "мб", "гб", "тб"
            },
        
            "вместимость": {
                "мл", "л", "см³", "дм³", "м³",
                "шт", "бутылок", "листов", "комплектов", "чел"
            },
        
            "емкость": {
                "мкл", "мл", "л",
                "см³", "дм³", "м³",
                "мкф", "нф", "пф", "ф",
                "мач", "ач",
                "вт·ч", "квт·ч",
                "мб", "гб", "тб",
                "шт"
            },
            "ёмкость": {
                "мкл", "мл", "л",
                "см³", "дм³", "м³",
                "мкф", "нф", "пф", "ф",
                "мач", "ач",
                "вт·ч", "квт·ч",
                "мб", "гб", "тб",
                "шт"
            },
        
            "резервуар": {"мл", "л", "см³", "дм³", "м³"},
            "бак": {"мл", "л", "см³", "дм³", "м³"},
            "объем бака": {"мл", "л", "см³", "дм³", "м³"},
            "объём бака": {"мл", "л", "см³", "дм³", "м³"},
        
            # Время
            "время": {"мкс", "мс", "с", "сек", "мин", "ч", "сут", "дни", "месяцы"},
            "длительность": {"мс", "с", "сек", "мин", "ч", "сут"},
            "продолжительность": {"мс", "с", "сек", "мин", "ч", "сут", "месяцы"},
        
            # Скорость
            "скорость": {
                "мм/с", "см/с", "м/с", "м/мин", "км/ч",
                "об/мин",
                "кадр/с", "стр/мин", "стежки/мин",
                "кб/с", "мб/с", "гб/с",
                "кбит/с", "мбит/с", "гбит/с",
                "iops", "ips"
            },
        
            "скорость вращения": {"об/мин", "об/с", "rpm"},
            "обороты": {"об/мин", "об/с", "rpm"},
            "число оборотов": {"об/мин", "об/с", "rpm"},
        
            "скорость передачи": {
                "бит/с", "кбит/с", "мбит/с", "гбит/с",
                "байт/с", "кб/с", "мб/с", "гб/с"
            },
            "скорость чтения": {"кб/с", "мб/с", "гб/с", "iops"},
            "скорость записи": {"кб/с", "мб/с", "гб/с", "iops"},
            "пропускная способность": {
                "бит/с", "кбит/с", "мбит/с", "гбит/с",
                "байт/с", "кб/с", "мб/с", "гб/с"
            },
            "битрейт": {"бит/с", "кбит/с", "мбит/с", "гбит/с"},
        
            # Расход / производительность
            "расход": {
                "мл/мин", "мл/ч",
                "л/мин", "л/ч", "л/сут",
                "м³/мин", "м³/ч", "м³/сут",
                "г/мин", "г/ч",
                "кг/мин", "кг/ч", "кг/сут",
                "л/кг", "л/м²", "кг/м²"
            },
        
            "производительность": {
                "г/мин", "г/ч",
                "кг/мин", "кг/ч", "кг/сут",
                "мл/мин", "л/мин", "л/ч",
                "м³/мин", "м³/ч", "м³/сут",
                "шт/мин", "шт/ч"
            },
        
            "расход воздуха": {"л/мин", "л/с", "м³/мин", "м³/ч", "cfm"},
            "расход воды": {"мл/мин", "л/мин", "л/ч", "м³/ч"},
            "расход топлива": {"л/ч", "л/100км", "кг/ч"},
            "воздушный поток": {"л/с", "л/мин", "м³/мин", "м³/ч", "cfm"},
        
            # Электрика
            "мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с.", "btu/h"},
            "потребляемая мощность": {"мвт", "вт", "квт", "ва", "ква"},
            "выходная мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с."},
        
            "напряжение": {"мв", "в", "кв"},
            "ток": {"мка", "ма", "а", "ка"},
            "сила тока": {"мка", "ма", "а", "ка"},
            "сопротивление": {"мом", "ом", "ком", "мом"},
            "заряд": {"мккл", "мкл", "кл", "мач", "ач"},
        
            "электрическая емкость": {"пф", "нф", "мкф", "мф", "ф"},
            "электрическая ёмкость": {"пф", "нф", "мкф", "мф", "ф"},
        
            # Энергия
            "энергия": {"дж", "кдж", "мдж", "вт·ч", "квт·ч", "кал", "ккал"},
            "энергопотребление": {"вт·ч", "квт·ч", "квт·ч/год"},
        
            # Частота
            "частота": {"гц", "кгц", "мгц", "ггц"},
            "частота вращения": {"об/мин", "об/с", "гц"},
            "частота обновления": {"гц"},
            "частота кадров": {"кадр/с", "fps"},
        
            # Температура
            "температура": {"°c", "°с", "°f", "k"},
        
            # Давление
            "давление": {
                "па", "кпа", "мпа",
                "бар", "мбар", "атм", "psi",
                "мм рт. ст.", "кгс/см²"
            },
        
            # Память / данные
            "память": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "объем памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "объём памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
            "оперативная память": {"мб", "гб", "тб"},
            "накопитель": {"мб", "гб", "тб"},
        
            "разрядность": {"бит"},
            "битность": {"бит"},
        
            # Оптика / свет
            "световой поток": {"лм"},
            "освещенность": {"лк"},
            "освещённость": {"лк"},
            "яркость": {"кд/м²", "нит"},
        
            # Свойства вещества
            "плотность": {
                "мг/мл", "г/мл", "г/л",
                "г/см³", "кг/л", "кг/м³"
            },
            "вязкость": {"па·с", "мпа·с", "сст", "cst"},
            "твердость": {"shore a", "shore d", "hb", "hrc", "hv"},
            "твёрдость": {"shore a", "shore d", "hb", "hrc", "hv"},
        
            # Проценты / относительные характеристики
            "процент": {"%"},
            "доля": {"%"},
            "концентрация": {"%", "мг/мл", "мг/л", "г/л", "моль/л"},
            "влажность": {"%"},
        
            # Шум
            "уровень шума": {"дб", "дба"},
            "шум": {"дб", "дба"},
        }
        
        
        self.unit_aliases = {
            "мм": {
                "мм",
                "мм.",
                "миллиметр",
                "миллиметра",
                "миллиметров",
                "в мм",
                "в мм.",
                "в миллиметрах",
                "в милиметрах",
                "в миллииметрах",
            },
        
            "см": {
                "см",
                "см.",
                "сантиметр",
                "сантиметра",
                "сантиметров",
                "в см",
                "в сантиметрах",
                "в сантиметрх",
                "в сантиментрах",
                "в сантимерах",
            },
        
            "м": {
                "м",
                "метр",
                "метра",
                "метров",
                "в м",
                "в метрах",
                "метра"
            },
        
            "дюйм": {
                '"',
                "дюйм",
                "дюймы",
                "дюймов",
                "в дюймах",
                "(дюйм)",
                "дюйма"
            },
        
            "мл": {
                "мл",
                "мл.",
                "миллилитр",
                "миллилитров",
                "миллилитра",
                "в мл",
                "в мл.",
                "в миллилитрах",
                "в милилитрах",
                "в миллиллитрах",
            },
        
            "л": {
                "л",
                "литр",
                "литров",
                "в л",
                "в литрах",
                "литра"
            },
        
            "г": {
                "г",
                "гр",
                "грамм",
                "граммов",
                "в г",
                "в гр",
                "в граммах",
            },
        
            "кг": {
                "кг",
                "килограмм",
                "килограмма",
                "килограммов",
                "в кг",
                "в килограммах",
                "в киллограммах",
                "в килогораммах",
            },
        
            "гб": {
                "гб",
                "гигабайт",
                "гигабайтов",
                "в гб",
                "в гигабайтах",
            },
        
            "мб": {
                "мб",
                "мегабайт",
                "мегабайтов",
                "в мб",
                "в мегабайтах",
            },
        
            "%": {
                "%",
                "процент",
                "процентов",
                "в %",
                "в процентах",
                "в массовых процентах",
            },
        }
        
        self.unit_aliases.update({
            "ч": {
                "ч",
                "час",
                "часа",
                "часов",
            },
            "мин": {
                "мин",
                "мин.",
                "минута",
                "минуты",
                "минут",
            },
            "с": {
                "с",
                "сек",
                "сек.",
                "секунда",
                "секунды",
                "секунд",
            },
            "вт": {
                "вт",
                "ватт",
                "ватта",
                "ваттов",
                "w",
            },
            "квт": {
                "квт",
                "киловатт",
                "киловатта",
                "киловаттов",
                "kw",
            },
        })
        
        self.extra_unit_aliases = {
            # Геометрия
            "нм": {
                "нм", "нм.",
                "нанометр", "нанометра", "нанометров",
                "nanometer", "nanometers", "nm",
            },
            "мкм": {
                "мкм", "мкм.",
                "микрометр", "микрометра", "микрометров",
                "микрон", "микрона", "микронов",
                "um", "µm", "μm",
            },
            "дм": {
                "дм", "дм.",
                "дециметр", "дециметра", "дециметров",
                "decimeter", "decimeters", "dm",
            },
            "км": {
                "км", "км.",
                "километр", "километра", "километров",
                "kilometer", "kilometers", "km",
            },
            "ft": {
                "ft", "foot", "feet",
                "фут", "фута", "футов",
            },
            "пог. м": {
                "пог. м", "пог.м", "пог м",
                "погонный метр", "погонного метра", "погонных метров",
            },
        
            # Угол
            "°": {
                "°",
                "градус", "градуса", "градусов",
                "deg", "degree", "degrees",
            },
            "рад": {
                "рад", "рад.",
                "радиан", "радиана", "радианов",
                "radian", "radians",
            },
        
            # Площадь
            "мм²": {
                "мм²", "мм2", "mm²", "mm2",
                "кв. мм", "кв.мм", "кв мм",
            },
            "см²": {
                "см²", "см2", "cm²", "cm2",
                "кв. см", "кв.см", "кв см",
            },
            "дм²": {
                "дм²", "дм2", "dm²", "dm2",
                "кв. дм", "кв.дм", "кв дм",
            },
            "м²": {
                "м²", "м2", "m²", "m2",
                "кв. м", "кв.м", "кв м",
            },
            "км²": {
                "км²", "км2", "km²", "km2",
                "кв. км", "кв.км", "кв км",
            },
            "га": {
                "га", "гектар", "гектара", "гектаров",
                "hectare", "hectares",
            },
        
            # Объём
            "мкл": {
                "мкл", "мкл.",
                "микролитр", "микролитра", "микролитров",
                "µl", "μl", "ul",
            },
            "мм³": {
                "мм³", "мм3", "mm³", "mm3",
                "куб. мм", "куб.мм", "куб мм",
            },
            "см³": {
                "см³", "см3", "cm³", "cm3",
                "куб. см", "куб.см", "куб см",
                "cc", "ccm",
            },
            "дм³": {
                "дм³", "дм3", "dm³", "dm3",
                "куб. дм", "куб.дм", "куб дм",
            },
            "м³": {
                "м³", "м3", "m³", "m3",
                "куб. м", "куб.м", "куб м",
            },
        
            # Масса
            "мг": {
                "мг", "мг.",
                "миллиграмм", "миллиграмма", "миллиграммов",
                "mg",
            },
            "т": {
                "т", "т.",
                "тонна", "тонны", "тонн",
                "ton", "tons", "tonne", "tonnes",
            },
            "lb": {
                "lb", "lbs",
                "фунт", "фунта", "фунтов",
                "pound", "pounds",
            },
            "oz": {
                "oz",
                "унция", "унции", "унций",
                "ounce", "ounces",
            },
            "карат": {
                "карат", "карата", "каратов",
                "ct", "carat", "carats",
            },
        
            # Сила / механика
            "н": {
                "н", "н.",
                "ньютон", "ньютона", "ньютонов",
                "newton", "newtons",
            },
            "кн": {
                "кн", "кн.",
                "килоньютон", "килоньютона", "килоньютонов",
                "kn",
            },
            "мн": {
                "мн",
                "меганьютон", "меганьютона", "меганьютонов",
            },
            "кгс": {
                "кгс", "kgf",
                "килограмм-сила", "килограмм силы",
            },
            "н·м": {
                "н·м", "н-м", "н*м",
                "n·m", "n-m", "n*m",
                "ньютон-метр",
            },
            "кгс·м": {
                "кгс·м", "кгс м", "кгс-м",
                "kgf·m", "kgf m",
            },
            "lb-ft": {
                "lb-ft", "lb ft", "lb·ft",
                "ft-lb", "ft lb",
            },
        
            # Время
            "мкс": {
                "мкс", "µs", "μs", "us",
                "микросекунда", "микросекунды", "микросекунд",
            },
            "мс": {
                "мс", "ms",
                "миллисекунда", "миллисекунды", "миллисекунд",
            },
            "сут": {
                "сут", "сут.",
                "сутки", "суток",
                "день", "дня", "дней",
                "day", "days",
            },
            "месяцы": {
                "месяц", "месяца", "месяцев", "месяцы",
                "month", "months",
            },
        
            # Скорость
            "мм/с": {"мм/с", "мм/сек", "mm/s", "mm/sec"},
            "см/с": {"см/с", "см/сек", "cm/s", "cm/sec"},
            "м/с": {"м/с", "м/сек", "m/s", "m/sec"},
            "м/мин": {"м/мин", "m/min"},
            "км/ч": {"км/ч", "км/час", "кмч", "km/h", "kmh", "kph"},
            "об/мин": {
                "об/мин", "об.мин", "об./мин",
                "оборот/мин", "оборотов/мин",
                "rpm", "r/min",
            },
            "об/с": {"об/с", "об/сек", "оборотов/с", "rps"},
            "кадр/с": {
                "кадр/с", "кадров/с",
                "кадр/сек", "кадров/сек",
                "fps",
            },
        
            # Передача данных
            "бит/с": {"бит/с", "бит/сек", "bps", "bit/s"},
            "кбит/с": {"кбит/с", "кбит/сек", "kbps", "kbit/s"},
            "мбит/с": {"мбит/с", "мбит/сек", "mbit/s", "mbps"},
            "гбит/с": {"гбит/с", "гбит/сек", "gbit/s", "gbps"},
            "кб/с": {"кб/с", "кб/сек", "kb/s", "kbyte/s"},
            "мб/с": {"мб/с", "мб/сек", "mb/s", "mbyte/s"},
            "гб/с": {"гб/с", "гб/сек", "gb/s", "gbyte/s"},
            "iops": {"iops"},
        
            # Расход
            "мл/мин": {"мл/мин", "ml/min"},
            "мл/ч": {"мл/ч", "ml/h", "ml/hr"},
            "л/мин": {"л/мин", "л/мин.", "l/min", "lpm"},
            "л/с": {"л/с", "л/сек", "l/s", "l/sec"},
            "л/ч": {"л/ч", "l/h", "l/hr"},
            "л/сут": {"л/сут", "л/сутки", "l/day"},
            "л/100км": {"л/100км", "л/100 км", "l/100km", "l/100 km"},
            "м³/мин": {"м³/мин", "м3/мин", "m³/min", "m3/min"},
            "м³/ч": {"м³/ч", "м3/ч", "m³/h", "m3/h"},
        
            # Количество
            "шт": {
                "шт", "шт.",
                "штук", "штука", "штуки",
                "ед", "ед.", "единица", "единицы", "единиц",
                "pcs", "pc",
            },
        
            # Мощность
            "мвт": {
                "мвт",
                "милливатт", "милливатта", "милливаттов",
            },
            "ва": {"ва", "va", "вольт-ампер", "вольт ампер"},
            "ква": {"ква", "kva", "киловольт-ампер", "киловольт ампер"},
            "л.с.": {
                "л.с.", "лс", "л. с.",
                "лошадиная сила", "лошадиных сил",
                "hp", "ps",
            },
        
            # Напряжение / ток
            "мв": {"мв", "mv", "милливольт", "милливольта", "милливольтов"},
            "в": {"в", "в.", "v", "вольт", "вольта", "вольтов"},
            "кв": {"кв", "kv", "киловольт", "киловольта", "киловольтов"},
            "мка": {"мка", "µa", "μa", "ua", "микроампер", "микроампера", "микроамперов"},
            "ма": {"ма", "ma", "миллиампер", "миллиампера", "миллиамперов"},
            "а": {"а", "а.", "a", "ампер", "ампера", "амперов"},
            "ка": {"ка", "ka", "килоампер", "килоампера", "килоамперов"},
        
            # Сопротивление
            "ом": {"ом", "ом.", "ohm", "Ω"},
            "ком": {"ком", "kohm", "kΩ", "килоом", "килоома", "килоомов"},
            "мом": {"мом", "mohm", "mΩ", "мегаом", "мегаома", "мегаомов"},
        
            # Ёмкость аккумулятора
            "мач": {
                "мач", "ма·ч", "ма*ч", "ма-ч",
                "mah", "ma·h", "ma-h",
                "миллиампер-час",
            },
            "ач": {
                "ач", "а·ч", "а*ч", "а-ч",
                "ah", "a·h", "a-h",
                "ампер-час",
            },
        
            # Электрическая ёмкость
            "пф": {"пф", "pf", "пикофарад", "пикофарада"},
            "нф": {"нф", "nf", "нанофарад", "нанофарада"},
            "мкф": {"мкф", "µf", "μf", "uf", "микрофарад", "микрофарада"},
            "мф": {"мф", "mf", "миллифарад", "миллифарада"},
            "ф": {"ф", "f", "фарад", "фарада"},
        
            # Энергия
            "дж": {"дж", "j", "джоуль", "джоуля", "джоулей"},
            "кдж": {"кдж", "kj", "килоджоуль", "килоджоуля", "килоджоулей"},
            "мдж": {"мдж", "mj", "мегаджоуль", "мегаджоуля", "мегаджоулей"},
            "вт·ч": {"вт·ч", "втч", "вт-ч", "вт*ч", "wh", "w·h", "w-h"},
            "квт·ч": {"квт·ч", "квтч", "квт-ч", "квт*ч", "kwh", "kw·h", "kw-h"},
        
            # Частота
            "гц": {"гц", "hz", "герц", "герца"},
            "кгц": {"кгц", "khz", "килогерц", "килогерца"},
            "мгц": {"мгц", "mhz", "мегагерц", "мегагерца"},
            "ггц": {"ггц", "ghz", "гигагерц", "гигагерца"},
        
            # Температура
            "°c": {
                "°c", "°с", "c°", "с°",
                "градус c", "градусов c",
                "градус цельсия", "градуса цельсия", "градусов цельсия",
                "celsius",
            },
            "°f": {
                "°f", "f°",
                "градус фаренгейта", "градусов фаренгейта",
                "fahrenheit",
            },
        
            # Давление
            "па": {"па", "pa", "паскаль", "паскаля", "паскалей"},
            "кпа": {"кпа", "kpa", "килопаскаль", "килопаскаля", "килопаскалей"},
            "мпа": {"мпа", "mpa", "мегапаскаль", "мегапаскаля", "мегапаскалей"},
            "бар": {"бар", "bar", "бара", "баров"},
            "мбар": {"мбар", "mbar", "миллибар", "миллибара", "миллибаров"},
            "атм": {"атм", "атм.", "атмосфера", "атмосферы", "атмосфер", "atm"},
            "psi": {"psi", "пси"},
        
            # Память
            "бит": {"бит", "бита", "битов", "bit", "bits"},
            "байт": {"байт", "байта", "байтов", "byte", "bytes"},
            "кб": {"кб", "kb", "kbyte", "килобайт", "килобайта", "килобайтов"},
            "тб": {"тб", "tb", "терабайт", "терабайта", "терабайтов"},
        
            # Свет
            "лм": {"лм", "lm", "люмен", "люмена", "люменов"},
            "лк": {"лк", "lx", "люкс", "люкса", "люксов"},
            "кд/м²": {"кд/м²", "кд/м2", "cd/m²", "cd/m2"},
            "нит": {"нит", "нита", "нитов", "nit", "nits"},
        
            # Шум
            "дб": {"дб", "db", "децибел", "децибела", "децибелов"},
            "дба": {"дба", "dba", "db(a)"},
        }

        # Сохраняем текущую логику расширения aliases.
        self.unit_aliases.update(
            self.extra_unit_aliases
        )

        self.alias_to_unit = {
            alias.lower().strip(): canonical_unit
            for canonical_unit, aliases
            in self.unit_aliases.items()
            for alias in aliases
        }

        self.skip_keywords = (
            "артикул",
            "артикулы",
            "oem",
            "sku",
            "part number",
            "partnumber",
            "партномер",
            "номер детали",
            "номер запчасти",
            "код товара",
            "код детали",
            "код производителя",
            "каталожный номер",
        )




        # Русская каноническая ЕИ -> имя для Pint
        self.ru_to_pint = {
            # Длина
            "нм": "nanometer",
            "мкм": "micrometer",
            "мм": "millimeter",
            "см": "centimeter",
            "дм": "decimeter",
            "м": "meter",
            "км": "kilometer",
            "дюйм": "inch",
            "ft": "foot",
        
            # Масса
            "мг": "milligram",
            "г": "gram",
            "кг": "kilogram",
            "т": "metric_ton",
            "lb": "pound",
            "oz": "ounce",
        
            # Объём
            "мкл": "microliter",
            "мл": "milliliter",
            "л": "liter",
            "см³": "centimeter**3",
            "дм³": "decimeter**3",
            "м³": "meter**3",
        
            # Площадь
            "мм²": "millimeter**2",
            "см²": "centimeter**2",
            "м²": "meter**2",
        
            # Время
            "мс": "millisecond",
            "с": "second",
            "сек": "second",
            "мин": "minute",
            "ч": "hour",
            "сут": "day",
        
            # Мощность
            "мвт": "milliwatt",
            "вт": "watt",
            "квт": "kilowatt",
        
            # Напряжение
            "мв": "millivolt",
            "в": "volt",
            "кв": "kilovolt",
        
            # Ток
            "ма": "milliampere",
            "а": "ampere",
        
            # Частота
            "гц": "hertz",
            "кгц": "kilohertz",
            "мгц": "megahertz",
            "ггц": "gigahertz",
        
            # Давление
            "па": "pascal",
            "кпа": "kilopascal",
            "мпа": "megapascal",
            "бар": "bar",
            "атм": "atmosphere",
            "psi": "psi",
        
            # Энергия
            "дж": "joule",
            "кдж": "kilojoule",
            "вт·ч": "watt_hour",
            "квт·ч": "kilowatt_hour",
        
            # Сопротивление
            "ом": "ohm",
            "ком": "kiloohm",
            "мом": "megaohm",
        
            # Температура
            "°c": "degC",
            "°с": "degC",
            "°f": "degF",
            "к": "kelvin",
        }
        
        # Русская каноническая ЕИ -> имя для Pint
        self.ru_to_pint.update({
            # Геометрия
            "мм³": "millimeter**3",
            "км²": "kilometer**2",
            "га": "hectare",
        
            # Масса
            "карат": "carat",
        
            # Сила / механика
            "н": "newton",
            "кн": "kilonewton",
            "мн": "meganewton",
            "кгс": "kilogram_force",
            "н·м": "newton * meter",
            "кгс·м": "kilogram_force * meter",
            "lb-ft": "pound_force * foot",
        
            # Время
            "мкс": "microsecond",
        
            # Скорость
            "мм/с": "millimeter / second",
            "см/с": "centimeter / second",
            "м/с": "meter / second",
            "м/мин": "meter / minute",
            "км/ч": "kilometer / hour",
            "об/мин": "revolution / minute",
            "об/с": "revolution / second",
            "кадр/с": "1 / second",
        
            # Скорость передачи данных
            "бит/с": "bit / second",
            "кбит/с": "kilobit / second",
            "мбит/с": "megabit / second",
            "гбит/с": "gigabit / second",
            "кб/с": "kilobyte / second",
            "мб/с": "megabyte / second",
            "гб/с": "gigabyte / second",
        
            # Расход
            "мл/мин": "milliliter / minute",
            "мл/ч": "milliliter / hour",
            "л/мин": "liter / minute",
            "л/с": "liter / second",
            "л/ч": "liter / hour",
            "л/сут": "liter / day",
            "л/100км": "liter / (100 * kilometer)",
            "м³/мин": "meter**3 / minute",
            "м³/ч": "meter**3 / hour",
        
            # Мощность
            "ва": "volt_ampere",
            "ква": "kilovolt_ampere",
            "л.с.": "horsepower",
        
            # Ток
            "мка": "microampere",
            "ка": "kiloampere",
        
            # Ёмкость аккумулятора
            "мач": "milliampere * hour",
            "ач": "ampere * hour",
        
            # Электрическая ёмкость
            "пф": "picofarad",
            "нф": "nanofarad",
            "мкф": "microfarad",
            "мф": "millifarad",
            "ф": "farad",
        
            # Энергия
            "мдж": "megajoule",
        
            # Память
            "бит": "bit",
            "байт": "byte",
            "кб": "kilobyte",
            "тб": "terabyte",
        
            # Свет
            "лм": "lumen",
            "лк": "lux",
            "кд/м²": "candela / meter**2",
        })
        
        
        # Для каждого keyword: в какую ЕИ приводим.
        # Слева/справа всё остаётся русским.
        self.keyword_standard_unit = {
            "ширина": "мм",
            "высота": "мм",
            "длина": "мм",
            "глубина": "мм",
            "толщина": "мм",
            "диаметр": "мм",
            "радиус": "мм",
            "размер": "мм",
            "габарит": "мм",
            "периметр": "мм",
            "окружность": "мм",
            "клиренс": "мм",
            "дорожный просвет": "мм",
            "колея": "мм",
            "база": "мм",
            "размах": "мм",
            "ход": "мм",
            "шаг": "мм",
            "зазор": "мм",
        
            "площадь": "м²",
        
            "вес": "кг",
            "масса": "кг",
            "максимальный вес": "кг",
            "допустимый вес": "кг",
        
            "объем": "л",
            "объём": "л",
            "резервуар": "л",
            "бак": "л",
            "объем бака": "л",
            "объём бака": "л",
        
            "время": "с",
            "длительность": "с",
            "продолжительность": "с",
        
            "мощность": "вт",
            "потребляемая мощность": "вт",
            "выходная мощность": "вт",
        
            "напряжение": "в",
            "ток": "а",
            "сила тока": "а",
        
            "частота": "гц",
            "частота обновления": "гц",
        
            "давление": "па",
            "сопротивление": "ом",
            "энергия": "дж",
        
            "температура": "°c",
        }
        
        self.keyword_standard_unit.update({
            # Геометрия
            "угол": "°",
        
            # Механика
            "нагрузка": "н",
            "сила": "н",
            "усилие": "н",
            "крутящий момент": "н·м",
            "грузоподъемность": "кг",
            "грузоподъёмность": "кг",
        
            # Вместимость
            "вместимость": "л",
        
            # Скорость
            "скорость": "м/с",
            "скорость вращения": "об/мин",
            "обороты": "об/мин",
            "число оборотов": "об/мин",
        
            "скорость передачи": "бит/с",
            "скорость чтения": "байт/с",
            "скорость записи": "байт/с",
            "пропускная способность": "бит/с",
            "битрейт": "бит/с",
        
            # Расход
            "расход": "л/ч",
            "расход воздуха": "м³/ч",
            "расход воды": "л/ч",
            "расход топлива": "л/100км",
            "воздушный поток": "м³/ч",
        
            # Электрика
            "электрическая емкость": "ф",
            "электрическая ёмкость": "ф",
            "заряд": "ач",
        
            # Энергия
            "энергопотребление": "квт·ч",
        
            # Частота
            "частота вращения": "об/мин",
            "частота кадров": "кадр/с",
        
            # Память
            "память": "гб",
            "объем памяти": "гб",
            "объём памяти": "гб",
            "оперативная память": "гб",
            "накопитель": "гб",
        
            # Свет
            "световой поток": "лм",
            "освещенность": "лк",
            "освещённость": "лк",
            "яркость": "кд/м²",
        })

        self.dimension_keywords = (
            "размер",
            "размеры",
            "габарит",
            "габариты",
        )

        # ========================================================
        # PRECOMPUTE
        # ========================================================

        self._build_unit_patterns()
        self._build_standard_patterns()
        self._build_conversion_map()
        self._build_dimension_patterns()


    def _build_unit_patterns(self):
        """Компилирует regex для keywords, aliases и удаления единиц из названий атрибутов."""

        self._keyword_patterns = {
            keyword: re.compile(
                rf"(?<!\w){re.escape(keyword)}(?!\w)",
                flags=re.IGNORECASE,
            )
            for keyword in self.keyword_possible_units
        }

        self._unit_alias_lists = {
            canonical_unit: sorted(
                self.unit_aliases.get(
                    canonical_unit,
                    {canonical_unit},
                ),
                key=len,
                reverse=True,
            )
            for units
            in self.keyword_possible_units.values()
            for canonical_unit in units
        }

        self._keyword_unit_patterns = {}
        self._keyword_alias_to_unit = {}

        for (
            keyword,
            allowed_units,
        ) in self.keyword_possible_units.items():

            alias_map = {}

            for canonical_unit in allowed_units:
                for alias in self._unit_alias_lists[
                    canonical_unit
                ]:
                    alias_map[
                        alias.lower().strip()
                    ] = canonical_unit

            aliases = sorted(
                alias_map,
                key=len,
                reverse=True,
            )

            if aliases:
                self._keyword_unit_patterns[
                    keyword
                ] = re.compile(
                    r"(?<!\w)("
                    + "|".join(
                        map(re.escape, aliases)
                    )
                    + r")(?!\w)",
                    flags=re.IGNORECASE,
                )

            self._keyword_alias_to_unit[
                keyword
            ] = alias_map

        self._unit_cleanup_patterns = {}

        for (
            canonical_unit,
            aliases,
        ) in self._unit_alias_lists.items():

            alternatives = "|".join(
                map(
                    re.escape,
                    sorted(
                        aliases,
                        key=len,
                        reverse=True,
                    ),
                )
            )

            self._unit_cleanup_patterns[
                canonical_unit
            ] = (
                re.compile(
                    rf"\(\s*(?:{alternatives})\s*\)",
                    flags=re.IGNORECASE,
                ),
                re.compile(
                    rf"(?<!\w)(?:{alternatives})(?!\w)",
                    flags=re.IGNORECASE,
                ),
            )

        aliases = sorted(
            self.alias_to_unit,
            key=len,
            reverse=True,
        )

        self._explicit_comma_pattern = re.compile(
            r",\s*("
            + "|".join(map(re.escape, aliases))
            + r")(?=$|\s)",
            flags=re.IGNORECASE,
        )

        self._explicit_bracket_pattern = re.compile(
            r"\(\s*("
            + "|".join(map(re.escape, aliases))
            + r")\s*\)",
            flags=re.IGNORECASE,
        )

        self._space_pattern = re.compile(
            r"\s+"
        )


    def _build_standard_patterns(self):
        """Компилирует regex для поиска attributes, которые нужно приводить к стандартной единице."""

        sorted_keywords = sorted(
            self.keyword_standard_unit,
            key=len,
            reverse=True,
        )

        self._standard_keyword_patterns = [
            (
                keyword,
                re.compile(
                    rf"(?<!\w)"
                    rf"{re.escape(keyword)}"
                    rf"(?!\w)",
                    flags=re.IGNORECASE,
                ),
            )
            for keyword in sorted_keywords
        ]


    def _build_conversion_map(self):
        """Через Pint один раз строит быстрые scale/offset преобразования между совместимыми единицами."""

        ureg = UnitRegistry()

        self._conversion_map = {}

        for (
            from_unit,
            from_pint,
        ) in self.ru_to_pint.items():

            for (
                to_unit,
                to_pint,
            ) in self.ru_to_pint.items():

                try:
                    y0 = (
                        0 * ureg(from_pint)
                    ).to(
                        to_pint
                    ).magnitude

                    y1 = (
                        1 * ureg(from_pint)
                    ).to(
                        to_pint
                    ).magnitude

                    self._conversion_map[
                        (from_unit, to_unit)
                    ] = (
                        y1 - y0,
                        y0,
                    )

                except Exception:
                    pass


    def _build_dimension_patterns(self):
        """Компилирует regex для многомерных размеров AxB и AxBxC."""

        self.dimension_pattern = re.compile(
            r"^\s*"
            r"(\d+(?:[.,]\d+)?)"
            r"\s*[xх×*]\s*"
            r"(\d+(?:[.,]\d+)?)"
            r"(?:"
                r"\s*[xх×*]\s*"
                r"(\d+(?:[.,]\d+)?)"
            r")?"
            r"\s*"
            r"(мм|см|дм|м|дюйм)?"
            r"\s*$",
            flags=re.IGNORECASE,
        )

        self.dimension_unit_pattern = re.compile(
            r"(?<!\w)"
            r"(мм|см|дм|м|дюйм)"
            r"(?!\w)",
            flags=re.IGNORECASE,
        )


    def normalize_physical_unit_pair(
        self,
        old_key,
        old_value,
    ):
        """Определяет единицу физической величины и приводит название attribute к форме `attribute, unit`."""

        key = str(old_key).strip()
        value = str(old_value).strip()

        key_lower = key.lower()

        if any(
            word in key_lower
            for word in self.skip_keywords
        ):
            return key, value

        # 1. keyword + подходящая единица.
        for (
            keyword,
            keyword_pattern,
        ) in self._keyword_patterns.items():

            if (
                keyword_pattern.search(key)
                is None
            ):
                continue

            unit_pattern = (
                self._keyword_unit_patterns.get(
                    keyword
                )
            )

            if unit_pattern is None:
                continue

            # Сначала ЕИ в value.
            match = unit_pattern.search(value)

            if match is not None:
                alias = (
                    match.group(1)
                    .lower()
                    .strip()
                )

                canonical_unit = (
                    self._keyword_alias_to_unit[
                        keyword
                    ][alias]
                )

                new_value = (
                    value[:match.start()]
                    + value[match.end():]
                ).strip()

                clean_key = key

                (
                    bracket_pattern,
                    standalone_pattern,
                ) = self._unit_cleanup_patterns[
                    canonical_unit
                ]

                clean_key = (
                    bracket_pattern.sub(
                        "",
                        clean_key,
                    )
                )

                clean_key = (
                    standalone_pattern.sub(
                        "",
                        clean_key,
                    )
                )

                clean_key = (
                    self._space_pattern
                    .sub(" ", clean_key)
                    .strip(" ,()")
                )

                return (
                    f"{clean_key}, {canonical_unit}",
                    new_value,
                )

            # Затем ЕИ в key.
            match = unit_pattern.search(key)

            if match is not None:
                alias = (
                    match.group(1)
                    .lower()
                    .strip()
                )

                canonical_unit = (
                    self._keyword_alias_to_unit[
                        keyword
                    ][alias]
                )

                (
                    bracket_pattern,
                    standalone_pattern,
                ) = self._unit_cleanup_patterns[
                    canonical_unit
                ]

                clean_key = (
                    bracket_pattern.sub(
                        "",
                        key,
                    )
                )

                clean_key = (
                    standalone_pattern.sub(
                        "",
                        clean_key,
                        count=1,
                    )
                )

                clean_key = (
                    self._space_pattern
                    .sub(" ", clean_key)
                    .strip(" ,()")
                )

                return (
                    f"{clean_key}, {canonical_unit}",
                    value,
                )

        # 2. Явное ", unit" / "(unit)".
        comma_match = (
            self._explicit_comma_pattern.search(
                key
            )
        )

        bracket_match = (
            self._explicit_bracket_pattern.search(
                key
            )
        )

        if comma_match is not None:
            match = comma_match
        elif bracket_match is not None:
            match = bracket_match
        else:
            return key, value

        alias = (
            match.group(1)
            .lower()
            .strip()
        )

        canonical_unit = (
            self.alias_to_unit[alias]
        )

        clean_key = (
            key[:match.start()].strip()
        )

        value_unit_pattern = re.compile(
            rf"(?<!\w)"
            rf"{re.escape(alias)}"
            rf"(?!\w)",
            flags=re.IGNORECASE,
        )

        new_value = value_unit_pattern.sub(
            "",
            value,
            count=1,
        ).strip()

        return (
            f"{clean_key}, {canonical_unit}",
            new_value,
        )


    def convert_physical_value(
        self,
        value,
        from_unit,
        to_unit,
    ):
        """Переводит одно числовое значение между двумя заранее известными совместимыми единицами."""

        try:
            number = float(
                str(value).replace(",", ".")
            )

            if from_unit == to_unit:
                return number

            conversion = (
                self._conversion_map.get(
                    (from_unit, to_unit)
                )
            )

            if conversion is None:
                return None

            scale, offset = conversion

            return (
                number * scale
                + offset
            )

        except Exception:
            return None


    def standardize_physical_unit_pair(
        self,
        key,
        value,
    ):
        """Приводит распознанную физическую величину к стандартной единице для соответствующего attribute."""

        if "," not in key:
            return key, value

        attr, unit = map(
            str.strip,
            key.rsplit(",", 1),
        )

        matched_keyword = None

        for (
            keyword,
            pattern,
        ) in self._standard_keyword_patterns:

            if pattern.search(attr):
                matched_keyword = keyword
                break

        if matched_keyword is None:
            return key, value

        target_unit = (
            self.keyword_standard_unit[
                matched_keyword
            ]
        )

        converted = self.convert_physical_value(
            value,
            unit.lower(),
            target_unit,
        )

        if converted is None:
            return key, value

        if converted.is_integer():
            converted = int(converted)

        return (
            f"{attr}, {target_unit}",
            str(converted),
        )


    def normalize_physical_attributes_dict(
        self,
        attrs,
    ):
        """Нормализует и стандартизирует все одномерные физические attributes одного словаря."""

        result = {}

        for (
            old_key,
            old_value,
        ) in attrs.items():

            key, value = (
                self.normalize_physical_unit_pair(
                    old_key,
                    old_value,
                )
            )

            key, value = (
                self.standardize_physical_unit_pair(
                    key,
                    value,
                )
            )

            result[key] = value

        return result


    def contains_standalone_keyword(
        self,
        text,
        keyword,
    ):
        """Проверяет наличие keyword как самостоятельного фрагмента, а не части другого слова."""

        return re.search(
            rf"(?<!\w)"
            rf"{re.escape(keyword)}"
            rf"(?!\w)",
            str(text),
            flags=re.IGNORECASE,
        ) is not None


    def normalize_multidimensional_attributes_dict(
        self,
        attrs,
    ):
        """Разбивает размеры AxB/AxBxC на длину, ширину и высоту и стандартно переводит их единицы."""

        result = {}

        for (
            old_key,
            old_value,
        ) in attrs.items():

            key = str(old_key).strip()
            value = str(old_value).strip()

            keyword = next(
                (
                    keyword
                    for keyword
                    in self.dimension_keywords
                    if self.contains_standalone_keyword(
                        key,
                        keyword,
                    )
                ),
                None,
            )

            if keyword is None:
                result[key] = value
                continue

            match = (
                self.dimension_pattern.fullmatch(
                    value
                )
            )

            if match is None:
                result[key] = value
                continue

            (
                a,
                b,
                c,
                value_unit,
            ) = match.groups()

            key_unit_match = (
                self.dimension_unit_pattern.search(
                    key
                )
            )

            unit = (
                value_unit.lower()
                if value_unit is not None
                else (
                    key_unit_match
                    .group(1)
                    .lower()
                    if key_unit_match
                    is not None
                    else None
                )
            )

            if unit is None:
                result[key] = value
                continue

            base_key = re.sub(
                r"\s*\(?\s*"
                r"длина\s*[xх×*]\s*ширина"
                r"(?:\s*[xх×*]\s*высота)?"
                r"\s*,?\s*"
                r"(?:мм|см|дм|м|дюйм)?"
                r"\s*\)?\s*$",
                "",
                key,
                flags=re.IGNORECASE,
            ).strip()

            base_key = re.sub(
                r"\s*(?:,\s*|\(\s*)"
                r"(?:мм|см|дм|м|дюйм)"
                r"\s*\)?\s*$",
                "",
                base_key,
                flags=re.IGNORECASE,
            ).strip()

            base_key = (
                base_key
                .replace("(", "")
                .replace(
                    " длина х ширина х высота",
                    "",
                )
                .replace(
                    " длина x ширина x высота",
                    "",
                )
                .strip()
            )

            for (
                replacement,
                dimension_value,
            ) in zip(
                (
                    "длина",
                    "ширина",
                    "высота",
                ),
                (a, b, c),
            ):
                if dimension_value is None:
                    continue

                new_key = re.sub(
                    rf"(?<!\w)"
                    rf"{re.escape(keyword)}"
                    rf"(?!\w)",
                    replacement,
                    base_key,
                    count=1,
                    flags=re.IGNORECASE,
                )

                (
                    new_key,
                    new_value,
                ) = (
                    self.standardize_physical_unit_pair(
                        f"{new_key}, {unit}",
                        dimension_value.replace(
                            ",",
                            ".",
                        ),
                    )
                )

                result[new_key] = new_value

        return result


    def normalize(self, attrs):
        """Применяет многомерную, затем обычную физическую нормализацию к одному словарю attributes."""

        attrs = (
            self.normalize_multidimensional_attributes_dict(
                attrs
            )
        )

        attrs = (
            self.normalize_physical_attributes_dict(
                attrs
            )
        )

        return attrs

## AttributeNameNormalizer

Нормализует названия атрибутов через морфологически согласованную замену синонимов.

Что делает:

- приводит название атрибута к базовому текстовому формату;
- лемматизирует русские слова;
- заменяет известные синонимы на единый вариант;
- сохраняет падеж, число и род заменяемого слова;
- кэширует уже обработанные слова;
- предотвращает коллизии названий атрибутов после переименования.

In [10]:
import re

import pymorphy3


class AttributeNameNormalizer:
    """
    Нормализует синонимы в названиях атрибутов,
    сохраняя грамматическую форму заменяемых слов.
    """

    def __init__(
        self,
        synonyms_df,
    ):
        """Строит карту синонимов, морфологический анализатор и кэш слов."""

        # synonym lemma -> replacer lemma
        self.replace_map = {
            synonym: row.replacer
            for row in synonyms_df.itertuples(
                index=False
            )
            for synonym in row.synonyms
        }

        self.morph = pymorphy3.MorphAnalyzer()

        self._synonym_word_cache = {}

        self._attribute_name_translation = (
            str.maketrans({
                "ё": "е",
                "-": " ",
                "(": "",
                ")": "",
            })
        )


    def normalize_synonym_word(
        self,
        word,
    ):
        """Заменяет одно слово на канонический синоним с сохранением его падежа, числа и рода."""

        word = word.lower()

        if word in self._synonym_word_cache:
            return self._synonym_word_cache[
                word
            ]

        source = self.morph.parse(
            word
        )[0]

        lemma = source.normal_form

        replacer = self.replace_map.get(
            lemma
        )

        if replacer is None:
            result = word

        else:
            target = self.morph.parse(
                replacer
            )[0]

            grammemes = {
                x
                for x in (
                    source.tag.case,
                    source.tag.number,
                    source.tag.gender,
                )
                if x is not None
            }

            inflected = target.inflect(
                grammemes
            )

            result = (
                inflected.word
                if inflected is not None
                else replacer
            )

        self._synonym_word_cache[
            word
        ] = result

        return result


    def _clean_attribute_name(
        self,
        text,
    ):
        """Приводит название атрибута к базовому формату без синонимической замены."""

        return (
            str(text)
            .lower()
            .translate(
                self._attribute_name_translation
            )
            .replace(
                "страна/регион",
                "страна производитель",
            )
        )


    def normalize_attribute_name_synonyms(
        self,
        text,
    ):
        """Очищает название атрибута и заменяет русские слова через словарь синонимов."""

        text = self._clean_attribute_name(
            text
        )

        return re.sub(
            r"[а-яе]+",
            lambda m: (
                self.normalize_synonym_word(
                    m.group()
                )
            ),
            text,
        )


    def normalize_synonym_attributes_dict(
        self,
        attrs,
    ):
        """Нормализует названия всех атрибутов словаря и предотвращает коллизии после синонимических замен."""

        normalized = {}

        original_attrs = {
            self._clean_attribute_name(
                attr
            )
            for attr in attrs
        }

        for attr, value in attrs.items():
            clean_attr = (
                self._clean_attribute_name(
                    attr
                )
            )

            new_attr = (
                self.normalize_attribute_name_synonyms(
                    attr
                )
            )

            # Не применяем синоним, если он создаёт
            # коллизию с существующим атрибутом.
            if (
                new_attr != clean_attr
                and (
                    new_attr in original_attrs
                    or new_attr in normalized
                )
            ):
                new_attr = clean_attr

            if new_attr not in normalized:
                normalized[
                    new_attr
                ] = value

        return normalized


    def normalize(
        self,
        attrs,
    ):
        """Нормализует названия атрибутов одного словаря."""

        return (
            self.normalize_synonym_attributes_dict(
                attrs
            )
        )

## ProductAttributesNormalizer

Объединяет все этапы нормализации атрибутов одной товарной карточки.

Порядок обработки:

1. JSON → словарь атрибутов;
2. нормализация многомерных размеров;
3. нормализация и стандартизация физических величин;
4. нормализация названий атрибутов через синонимы;
5. словарь → JSON.

Ошибка отдельного этапа не прерывает обработку остальных этапов.

In [11]:
import json


class ProductAttributesNormalizer:
    def __init__(
        self,
        physical_normalizer,
        attribute_name_normalizer,
        blacklist=None,
    ):
        self.physical = physical_normalizer
        self.attribute_names = attribute_name_normalizer

        self.blacklist = {
            x.casefold()
            for x in (blacklist or [])
        }

    def _drop_blacklisted_attributes(
        self,
        attrs,
    ):
        return {
            key: value
            for key, value in attrs.items()
            if str(key).casefold() not in self.blacklist
        }

    
    def normalize_dict(
        self,
        attrs,
    ):
        try:
            attrs = (
                self.physical
                .normalize_multidimensional_attributes_dict(
                    attrs
                )
            )
        except Exception:
            pass
    
        try:
            attrs = (
                self.physical
                .normalize_physical_attributes_dict(
                    attrs
                )
            )
        except Exception:
            pass
    
        try:
            attrs = (
                self.attribute_names
                .normalize_synonym_attributes_dict(
                    attrs
                )
            )
        except Exception:
            pass
    
        attrs = self._drop_blacklisted_attributes(
            attrs
        )
    
        return attrs

    def normalize(
        self,
        raw,
    ):
        """Разбирает JSON attributes, применяет полный pipeline и возвращает нормализованный JSON."""

        try:
            attrs = json.loads(raw)
        except Exception:
            return raw

        attrs = self.normalize_dict(
            attrs
        )

        try:
            return json.dumps(
                attrs,
                ensure_ascii=False,
            )
        except Exception:
            return raw

## Код вызова нормализации

In [12]:
"""
cards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human.parquet")
labels = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/matches.parquet")
synonyms_df = pd.read_pickle("/kaggle/input/datasets/kehhill/ozon-e-cup/synonyms_df.pkl")
# cards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human_normalized.parquet")

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 1000)

physical_normalizer = (
    PhysicalUnitNormalizer()
)

attribute_name_normalizer = (
    AttributeNameNormalizer(
        synonyms_df=synonyms_df,
    )
)

normalizer = ProductAttributesNormalizer(
    physical_normalizer=physical_normalizer,
    attribute_name_normalizer=(
        attribute_name_normalizer
    ),
)


# Пример
normalized = normalizer.normalize(
    cards["attributes"].iloc[0]
)
print()
"""

'\ncards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human.parquet")\nlabels = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/matches.parquet")\nsynonyms_df = pd.read_pickle("/kaggle/input/datasets/kehhill/ozon-e-cup/synonyms_df.pkl")\n# cards = pd.read_parquet("/kaggle/input/datasets/kehhill/ozon-e-cup/items_human_normalized.parquet")\n\npd.set_option(\'display.max_rows\', 100)\npd.set_option(\'display.max_colwidth\', 1000)\n\nphysical_normalizer = (\n    PhysicalUnitNormalizer()\n)\n\nattribute_name_normalizer = (\n    AttributeNameNormalizer(\n        synonyms_df=synonyms_df,\n    )\n)\n\nnormalizer = ProductAttributesNormalizer(\n    physical_normalizer=physical_normalizer,\n    attribute_name_normalizer=(\n        attribute_name_normalizer\n    ),\n)\n\n\n# Пример\nnormalized = normalizer.normalize(\n    cards["attributes"].iloc[0]\n)\nprint()\n'

## ProductParser

Полный пайплайн обработки товарной карточки.

Объединяет:

- извлечение атрибутов из названия через NER и алгоритмический парсер;
- нормализацию исходных атрибутов карточки;
- объединение исходных и извлечённых атрибутов.

Работает как с одной карточкой, так и с батчем.

In [13]:
class ProductAttributesProcessor:
    """
    Полный pipeline обработки атрибутов товара:

    1. нормализует исходные attributes карточки;
    2. извлекает attributes из name;
    3. нормализует физические величины, найденные в name;
    4. дополняет карточку найденными attributes,
       не перезаписывая уже существующие.
    """

    def __init__(
        self,
        name_parser,
        attributes_normalizer,
        physical_normalizer,
    ):
        self.name_parser = name_parser
        self.attributes_normalizer = attributes_normalizer
        self.physical_normalizer = physical_normalizer


    def _prepare_normalized_attributes(
        self,
        attributes,
    ):
        normalized = (
            self.attributes_normalizer.normalize(
                attributes
            )
        )

        if isinstance(normalized, str):
            try:
                normalized = json.loads(normalized)
            except Exception:
                normalized = {}

        return normalized


    def _normalize_parsed_name(
        self,
        parsed_name,
    ):
        """
        Прогоняет найденные в name физические attributes
        через обычную physical normalization + Pint.
        """

        return (
            self.physical_normalizer
            .normalize_physical_attributes_dict(
                parsed_name
            )
        )


    def process(
        self,
        name,
        attributes,
    ):
        normalized_attributes = (
            self._prepare_normalized_attributes(
                attributes
            )
        )

        parsed_name = self.name_parser.parse(
            name
        )

        parsed_name = (
            self._normalize_parsed_name(
                parsed_name
            )
        )

        # Исходные attributes имеют приоритет.
        return {
            **parsed_name,
            **normalized_attributes,
        }


    def process_batch(
        self,
        names,
        attributes,
        batch_size=6000,
    ):
        names = list(names)
        attributes = list(attributes)

        # 1. Нормализация карточек.
        normalized_attributes = (
            self.attributes_normalizer
            .normalize_batch(attributes)
        )

        normalized_attributes = [
            (
                json.loads(attrs)
                if isinstance(attrs, str)
                else attrs
            )
            for attrs in normalized_attributes
        ]

        # 2. Парсинг name.
        parsed_names = (
            self.name_parser.parse_batch(
                names,
                batch_size=batch_size,
            )
        )

        # 3. Pint-нормализация физических
        #    attributes, найденных в name.
        parsed_names = [
            self._normalize_parsed_name(attrs)
            for attrs in parsed_names
        ]

        # 4. name только дополняет карточку.
        return [
            {
                **parsed,
                **normalized,
            }
            for parsed, normalized in zip(
                parsed_names,
                normalized_attributes,
            )
        ]

In [14]:
from joblib import Parallel, delayed


def _normalize_attributes_chunk_worker(
    normalizer,
    chunk,
):
    return [
        normalizer.normalize(attributes)
        for attributes in chunk
    ]


class ParallelProductAttributesNormalizer:
    """
    Параллельная CPU-обёртка над ProductAttributesNormalizer.
    """

    def __init__(
        self,
        normalizer,
        n_jobs=2,
        chunk_size=10_000,
    ):
        self.normalizer = normalizer
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size

    def normalize(self, attributes):
        return self.normalizer.normalize(
            attributes
        )

    def normalize_batch(self, attributes):
        attributes = list(attributes)

        if not attributes:
            return []

        if (
            self.n_jobs == 1
            or len(attributes) <= self.chunk_size
        ):
            return [
                self.normalizer.normalize(x)
                for x in attributes
            ]

        chunks = [
            attributes[i:i + self.chunk_size]
            for i in range(
                0,
                len(attributes),
                self.chunk_size,
            )
        ]

        parts = Parallel(
            n_jobs=self.n_jobs,
            backend="loky",
            pre_dispatch=self.n_jobs,
            batch_size=1,
        )(
            delayed(
                _normalize_attributes_chunk_worker
            )(
                self.normalizer,
                chunk,
            )
            for chunk in chunks
        )

        return [
            item
            for part in parts
            for item in part
        ]

## ProductAttributesPipeline

Высокоуровневый интерфейс для полного пайплайна обработки атрибутов товарной карточки.

Что делает:

- загружает NER-модель, tokenizer и semantic cluster centers;
- загружает словарь синонимов;
- собирает `ProductNameParser`;
- собирает параллельный парсер физических характеристик;
- собирает нормализацию исходных `attributes`;
- включает параллельную нормализацию атрибутов;
- объединяет всё в `ProductAttributesProcessor`.

Основной способ создания — `ProductAttributesPipeline.from_pretrained(...)`.

Основные параметры:

- `model_dir` — директория NER-модели;
- `cluster_centers_path` — semantic cluster centers;
- `synonyms_df_path` — таблица синонимов;
- `n_jobs` — число CPU-процессов;
- `chunk_size` — размер CPU-chunk;
- `batch_size` — размер GPU batch для NER;
- `max_length` — максимальная длина токенизированного названия;
- `device` — устройство для нейросети;
- `use_amp` — использование mixed precision на GPU.

После создания внешний API состоит из двух методов:

- `process(name, attributes)` — обработка одной карточки;
- `process_batch(names, attributes)` — обработка батча.

Все промежуточные модели, парсеры, нормализаторы и настройки их взаимодействия создаются внутри класса.

In [15]:
class ProductAttributesPipeline:
    """
    Собирает полный pipeline:
    name parsing + attributes normalization.
    """

    DEFAULT_CLASS_NAMES = (
        "O",
        "бренд",
        "тип",
        "модель",
        "материал",
        "цвет",
        "размер",
        "назначение",
        "артикул",
    )

    @classmethod
    def from_pretrained(
        cls,
        model_dir,
        cluster_centers_path,
        synonyms_df_path,
        *,
        blacklist=None,
        n_jobs=2,
        chunk_size=5000,
        batch_size=6000,
        max_length=100,
        device=None,
        use_amp=True,
        class_names=None,
    ):
        if device is None:
            device = torch.device(
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        if class_names is None:
            class_names = cls.DEFAULT_CLASS_NAMES

        synonyms_df = pd.read_pickle(
            synonyms_df_path
        )

        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            use_fast=True,
        )

        model = WordNERModel.from_pretrained_dir(
            model_dir,
            num_classes=len(class_names),
            device=device,
        )

        ner_postprocessor = NerPostprocessor(
            model=model,
            tokenizer=tokenizer,
            cluster_centers_path=cluster_centers_path,
            class_names=class_names,
            device=device,
            max_length=max_length,
            use_amp=use_amp,
        )

        physical_parser = (
            ParallelPhysicalAttributeParser(
                n_jobs=n_jobs,
                chunk_size=chunk_size,
            )
        )

        name_parser = ProductNameParser(
            ner_postprocessor=ner_postprocessor,
            physical_parser=physical_parser,
        )

        physical_normalizer = (
            PhysicalUnitNormalizer()
        )

        attribute_name_normalizer = (
            AttributeNameNormalizer(
                synonyms_df=synonyms_df,
            )
        )

        base_attributes_normalizer = ProductAttributesNormalizer(
            physical_normalizer=physical_normalizer,
            attribute_name_normalizer=attribute_name_normalizer,
            blacklist=blacklist,
        )
        
        attributes_normalizer = (
            ParallelProductAttributesNormalizer(
                normalizer=(
                    base_attributes_normalizer
                ),
                n_jobs=n_jobs,
                chunk_size=chunk_size,
            )
        )

        processor = ProductAttributesProcessor(
            name_parser=name_parser,
            attributes_normalizer=attributes_normalizer,
            physical_normalizer=physical_normalizer,
        )

        return cls(
            processor=processor,
            batch_size=batch_size,
        )

    def __init__(
        self,
        processor,
        batch_size=6000,
    ):
        self.processor = processor
        self.batch_size = batch_size

    def process(
        self,
        name,
        attributes,
    ):
        return self.processor.process(
            name=name,
            attributes=attributes,
        )

    def process_batch(
        self,
        names,
        attributes,
        batch_size=None,
    ):
        if batch_size is None:
            batch_size = self.batch_size

        return self.processor.process_batch(
            names=names,
            attributes=attributes,
            batch_size=batch_size,
        )

## Пример работы

In [16]:
# from product_attributes import (
#     ProductAttributesPipeline,
# )
import json

import torch
from transformers import AutoTokenizer


BLACKLIST = {
    "комплект",
    "модель цвета"
}

pipeline = ProductAttributesPipeline.from_pretrained(
    model_dir="/kaggle/input/datasets/kehhill/rubert-v3-ner/rubert_tiny2_word_ner",
    cluster_centers_path="/kaggle/input/datasets/kehhill/ozon-e-cup/cluster_centers.pt",
    synonyms_df_path="/kaggle/input/datasets/kehhill/ozon-e-cup/synonyms_df.pkl",
    blacklist=BLACKLIST,
    n_jobs=2,
    chunk_size=5000,
    batch_size=6000,
)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
names = [
    "bosch дрель gsb 13 re 600 вт синяя",
    "стол деревянный 120x60x75 см дуб",
    "чайник redmond rk-m170s 1.7 л черный",
]

attributes = [
    '{"бренд": "Bosch", "мощность": "600 Вт"}',
    '{"материал изделия": "дерево", "размер": "120x60x75 см"}',
    '{"производитель": "Redmond", "объем": "1.7 литра"}',
]

results = pipeline.process_batch(
    names=names,
    attributes=attributes,
    batch_size=6000,
)

display(results)

[{'бренд': 'Bosch',
  'тип': 'дрель',
  'модель': 'gsb 13 re 600',
  'цвет': 'синяя',
  'мощность, вт': '600'},
 {'тип': 'стол деревянный',
  'цвет': 'дуб',
  'длина, мм': '1200',
  'ширина, мм': '600',
  'высота, мм': '750',
  'материал изделия': 'дерево'},
 {'тип': 'чайник',
  'бренд': 'redmond',
  'модель': 'rk-m170s',
  'цвет': 'черный',
  'объем, л': '1.7',
  'производитель': 'Redmond'}]